<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase3/phase3_02_offline_engine_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Phase 3 · Sector 2 · Offline engine tests

On a fresh Colab VM, need to upload `phase3_02_grounded_question_generation.ipynb` to the Files panel (`/content`), or set **MAIN_NOTEBOOK** to its path on mounted storage. This test notebook loads the engine definitions from that file. The main notebook does not need to be executed.

With **CONTRACT_SOURCE = "github"**, setup resolves the current main revision of `bogdanparvu18/msc-graduate-project`, downloads the fourteen V13 contracts through the production loader and checks their local bytes against the manifest. The setup cell prints the revision and VM directory. With **CONTRACT_SOURCE = "local"**, specify an already available directory using **LOCAL_CONTRACT_DIR** to run without network access from the start.


After all tests pass, Section 9 saves the accepted examples from the two-wording campaign to `/content/mmvqa_offline_qa_examples.csv` in Colab. Open **Files → Refresh** to see it. The CSV stays on this VM until the runtime is deleted. Download it from the Files panel to keep a copy.


In [1]:
import ast
import copy
import csv
import datetime as dt
import errno
import gzip
import hashlib
import inspect
import io
import itertools
import json
import math
import os
import re
import shutil
import socket
import sqlite3
import tarfile
import tempfile
import unicodedata
import uuid
from collections import Counter, defaultdict
from contextlib import ExitStack, contextmanager
from functools import lru_cache
from http.client import IncompleteRead
from importlib.metadata import version as package_version
from io import BytesIO
from pathlib import Path, PurePosixPath
from types import ModuleType
from typing import Callable
from unittest.mock import Mock, patch
from urllib.error import HTTPError
from urllib.parse import quote, unquote, urlparse
from urllib.request import HTTPRedirectHandler, Request, build_opener, urlopen

from jsonschema import Draft202012Validator
# S8 progress callbacks and a renderer spy exercise the display path without a Colab frontend.
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(**kwargs):
        return Mock()


In [2]:
MAIN_NOTEBOOK = "phase3_02_grounded_question_generation.ipynb"  # @param {type:"string"}
CONTRACT_SOURCE = "github"  # @param ["github", "local"]
# GitHub setup can reuse the production VM cache. Leave blank for the production default.
CONTRACT_CACHE_DIR = ""  # @param {type:"string"}
# Used only with the local source.
LOCAL_CONTRACT_DIR = "v13"  # @param {type:"string"}
# Optional explicit trusted manifest for either source.
CONTRACT_MANIFEST_PATH = ""  # @param {type:"string"}


### 1 · Load the production components

Only the named engine cells are compiled. The production notebook's setup, inventory and real generation entry cells are never executed by this loader. The tests inject local evidence, mock API responses and temporary databases at the corresponding boundaries.


In [3]:
@contextmanager
def count_file_hashes():
    """Count SHA-256 evaluations by content, including streamed and single-buffer hashing."""
    counts = Counter()
    implementation = hashlib.sha256

    class TrackedDigest:
        def __init__(self, data=b"", **kwargs):
            self.digest_object = implementation(data, **kwargs)
            self.recorded = False

        def update(self, data):
            self.digest_object.update(data)
            self.recorded = False

        def hexdigest(self):
            value = self.digest_object.hexdigest()
            if not self.recorded:
                counts[value] += 1
                self.recorded = True
            return value

        def digest(self):
            self.hexdigest()
            return self.digest_object.digest()

    with patch.object(hashlib, "sha256", TrackedDigest):
        yield counts


def run_import_layout_regressions(path):
    """Locate executable top-level calls in the same order as Colab Run all."""
    cells = json.loads(path.read_bytes())["cells"]
    calls = defaultdict(list)
    section = ""
    mode_controls = []
    for index, cell in enumerate(cells):
        body = "".join(cell["source"])
        if cell["cell_type"] == "markdown":
            title = next((line for line in body.splitlines() if re.match(r"^#+\s*\d+\s*·", line)), None)
            if title:
                section = title
            continue
        if cell["cell_type"] != "code" or body.lstrip().startswith("%%"):
            continue
        for line in body.splitlines():
            control = re.fullmatch(r'MODE\s*=\s*"[^"]+"\s*#\s*@param\s+(\[.*\])\s*', line)
            if control:
                mode_controls.append(json.loads(control.group(1)))
        for node in ast.walk(ast.parse(body)):
            if isinstance(node, ast.Call):
                target = node.func
                if isinstance(target, ast.Name):
                    calls[target.id].append((index, section))
                elif isinstance(target, ast.Subscript) and isinstance(target.slice, ast.Constant):
                    calls[target.slice.value].append((index, section))
    imports = calls["import_sector2_inputs"]
    runs = calls["execute_sector2"]
    if len(imports) != 1 or not re.match(r"^#+\s*1\s*·", imports[0][1]):
        raise AssertionError("Section 1 must actually execute the consolidated import once.")
    if len(runs) != 1 or imports[0][0] >= runs[0][0]:
        raise AssertionError("Import must execute before census or generation.")
    if mode_controls != [["inventory_only", "real_openai"]]:
        raise AssertionError("The Colab mode control must offer inventory_only and real_openai.")
    return {"checks_passed": 3, "checks": ["Section 1 executes the consolidated source import exactly once",
                                           "The generation cell appears after the completed import",
                                           "The Colab control exposes exactly the two supported modes"]}


PRODUCTION_COMPONENTS = (
    "sources", "real", "evidence", "api", "validation", "sampling",
    "workflow", "inventory", "generation_campaign",
)

@contextmanager
def offline_only():
    def block(*args, **kwargs):
        raise AssertionError("Offline engine tests attempted a network connection.")
    with ExitStack() as guards:
        guards.enter_context(patch.object(socket, "getaddrinfo", block))
        guards.enter_context(patch.object(socket.socket, "connect", block))
        guards.enter_context(patch.object(socket.socket, "connect_ex", block))
        yield

def read_components(path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Place the production notebook at {path}, or set MAIN_NOTEBOOK to its actual name."
        )
    raw = path.read_bytes()
    document = json.loads(raw)
    found = {}
    for cell in document["cells"]:
        if cell["cell_type"] != "code":
            continue
        header, _, body = "".join(cell["source"]).partition("\n")
        if not header.startswith("%%sector2_component "):
            continue
        name = header.split(maxsplit=1)[1].strip()
        if name not in PRODUCTION_COMPONENTS:
            raise ValueError(f"Unexpected component {name!r}. Use the companion production-only notebook.")
        if name in found or not body.strip():
            raise ValueError(f"Duplicate or empty production component: {name}")
        found[name] = body
    missing = set(PRODUCTION_COMPONENTS) - found.keys()
    if missing:
        raise ValueError(f"Missing production components in {path}: {sorted(missing)}.")
    return hashlib.sha256(raw).hexdigest(), found

production_path = Path(MAIN_NOTEBOOK).expanduser().resolve()
production_sha256, component_sources = read_components(production_path)
component_hashes = {
    name: hashlib.sha256(source.encode("utf-8")).hexdigest()
    for name, source in component_sources.items()
}

# Test replacements for optional production dependencies: no parquet reader or Colab secrets.
pq, userdata = None, None
with offline_only():
    for name in PRODUCTION_COMPONENTS:
        source_code = component_sources[name]
        exec(compile(source_code, f"<mmvqa-sector2/{name}>", "exec"), globals())
print("Production notebook:", production_path)
print("Production SHA-256:", production_sha256)


Production notebook: /content/phase3_02_grounded_question_generation.ipynb
Production SHA-256: 54b7f17154a75d70403e0be00b2a2aec7636b08c0e46cb70dc71c8cd1ab8e524


### Prepare the V13 contracts

The next cell calls `open_contract_release()` from the production notebook. In GitHub mode this function captures the current `main` commit, imports the fourteen V13 contracts to the VM and checks their SHA-256. The production loader also supplies the reviewed detached manifest when the published revision omits it. An explicit trusted manifest can be selected with **CONTRACT_MANIFEST_PATH**.

The same verified registry is used throughout the test run. Select **local** to test a directory containing the V13 files and its manifest entirely offline.


In [4]:
# The production loader owns all downloads, manifest selection, retries and file checks.
prepared_registry = None
if CONTRACT_SOURCE not in {"github", "local"}:
    raise ValueError("Select github or local for CONTRACT_SOURCE.")
default_work = Path("/content") if Path("/content").is_dir() else Path.cwd()
contract_cache = (Path(CONTRACT_CACHE_DIR).expanduser() if CONTRACT_CACHE_DIR
                  else default_work / "mmvqa_phase3_sector2" / "cache")
local_test_stage = None
if CONTRACT_SOURCE == "github":
    prepared_registry = open_contract_release(contract_cache, online=True, manifest_path=CONTRACT_MANIFEST_PATH or None)
else:
    local_test_stage = tempfile.TemporaryDirectory(prefix="mmvqa-v13-local-")
    with offline_only():
        prepared_registry = open_contract_release(local_test_stage.name, online=False,
                                                 local_contract_dir=Path(LOCAL_CONTRACT_DIR).expanduser(),
                                                 manifest_path=CONTRACT_MANIFEST_PATH or None)

preparation_manifest_source = prepared_registry["manifest_origin"]
print("Contract source:", CONTRACT_SOURCE)
if prepared_registry["registry_commit"]:
    print("GitHub revision:", prepared_registry["registry_commit"])
print("Manifest source:", preparation_manifest_source, "| SHA-256:", prepared_registry["manifest_sha256"])
print("Verified contracts:", len(prepared_registry["contract_sha256"]))
print("Verified directory:", prepared_registry["staged_directory"])
if CONTRACT_SOURCE == "github":
    sync = prepared_registry["contract_sync"]
    print("Reused on VM:", len(sync["reused_in_place"]) + len(sync["copied_from_vm"]))
    print("Downloaded contract files:", sync["downloaded_from_github"])


V13 detached manifest · batch 1 · files processed for transfer:   0%|          | 0/1 [00:00<?, ?file/s]

V13 detached manifest · batch 1 · verified VM files:   0%|          | 0/1 [00:00<?, ?file/s]

V13 contracts · batch 1 · files processed for transfer:   0%|          | 0/14 [00:00<?, ?file/s]

V13 contracts · batch 1 · verified VM files:   0%|          | 0/14 [00:00<?, ?file/s]

Contract source: github
GitHub revision: 8671d60736c047c2e4c7942e11aa6d3bc1592f8f
Manifest source: reviewed_detached_manifest | SHA-256: 6befcf2d5c6ff8f317cc2cb4e690363eb9f49a104e95c426ba83499df6aba068
Verified contracts: 14
Verified directory: /content/mmvqa_phase3_sector2/cache/contract_imports/v13-8671d60736c047c2e4c7942e11aa6d3bc1592f8f
Reused on VM: 0
Downloaded contract files: ['evidence_frame_localization.grammar.v13.json', 'evidence_frame_localization.semantics.v13.json', 'finding_category.grammar.v13.json', 'finding_category.semantics.v13.json', 'finding_identification.grammar.v13.json', 'finding_identification.semantics.v13.json', 'observed_anchor_order.grammar.v13.json', 'observed_anchor_order.semantics.v13.json', 'optional_spatial_localization.grammar.v13.json', 'optional_spatial_localization.semantics.v13.json', 'temporal_relative_evidence_localization.grammar.v13.json', 'temporal_relative_evidence_localization.semantics.v13.json', 'verified_reference_comparison.grammar.v1

### 2 · Eight synthetic profiles

Geometric test images and fabricated identifiers exercise the same evidence adapters and answer derivation used on real inputs. The generator receives only the public roles for the current case. Each wording fixture is reserved for an independent check.


In [5]:
# Synthetic observations use invented identifiers and schematic images.
# They exercise control flow. They are never evidence for a clinical corpus.

DEMO_PROFILES = {
    "finding_identification": "any_one_finding",
    "finding_category": "classify_named_finding",
    "evidence_frame_localization": "any_one_positive_display_unit",
    "optional_spatial_localization": "locate_one_bbox",
    "verified_reference_comparison": "select_normal_reference_unit",
    "observed_anchor_order": "select_earlier_anchor",
    "temporal_relative_evidence_localization": "one_positive_displayed_unit_relative_to_reference",
}
DEMO_CLASSES = {
    "demo:polyp": ("polyp", "mucosal_lesion"),
    "demo:ampulla": ("ampulla of Vater", "anatomical_landmark"),
    "demo:normal": ("normal clean mucosa", "normal"),
}


def demo_image_bytes(frame_index, marks):
    """A genuine, reproducible 96×64 PGM raster of geometric shapes; never a medical image."""
    width, height = 96, 64
    pixels = bytearray(20 + ((x + y + frame_index) % 25)
                       for y in range(height) for x in range(width))
    for x1, y1, x2, y2 in marks:
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        rx, ry = (x2 - x1) / 2, (y2 - y1) / 2
        for y in range(y1, y2):
            for x in range(x1, x2):
                if ((x - cx) / rx) ** 2 + ((y - cy) / ry) ** 2 <= 1:
                    pixels[y * width + x] = 180
    return b"P5\n96 64\n255\n" + pixels


def synthetic_evidence(task, landmark=False):
    """One privately annotated, complete test presentation for the specified profile."""
    if task not in DEMO_PROFILES or (landmark and task != "finding_identification"):
        raise ValueError("Unknown synthetic profile")
    profile = "any_one_anatomical_landmark" if landmark else DEMO_PROFILES[task]
    query = "demo:ampulla" if landmark else "demo:polyp"
    variant, state = "full", "label_without_box"
    entries = [(10, [query])]
    scope = "image_1"
    if task == "optional_spatial_localization":
        state = "label_with_box"
        entries = [(10, [query, query])]
    elif task == "verified_reference_comparison":
        variant, state = "comparison", "explicit_normal_reference"
        entries, scope = [(20, [query]), (40, ["demo:normal"])], "this comparison"
    elif task == "observed_anchor_order":
        variant, state = "temporal_multi", "multiple_verified_anchors"
        entries, scope = [(80, [query]), (30, [query])], "the displayed panels"
    elif task == "evidence_frame_localization":
        variant, state = "temporal_multi", "label_without_box"
        entries, scope = [(10, [query]), (20, []), (30, [query])], "the displayed panels"
    elif task == "temporal_relative_evidence_localization":
        variant, state = "temporal_multi", "unlabelled_neighbor"
        entries = [(10, [query]), (20, []), (30, [query]), (40, [query])]
        scope = "this sequence"

    # Separate fixture worlds, including physical frame and pixel identity.
    video_id = f"synthetic-{task}-{profile}"
    texture_seed = int(hashlib.sha256(video_id.encode()).hexdigest()[:6], 16)
    multi = len(entries) > 1
    units = []
    for index, (frame, classes) in enumerate(entries):
        unit_id = f"image_1/panel_{index + 1}" if multi else "image_1"
        label = f"panel {chr(65 + index)}" if multi else "image_1"
        marks = []
        observations = []
        for n, class_id in enumerate(classes):
            box = [12 + n * 40, 16, 36 + n * 40, 38]
            if class_id != "demo:normal":
                marks.append(box)
            observations.append({
                "id": f"demo-observation-{frame}-{n}",
                "class_id": class_id,
                "clinical_group": DEMO_CLASSES[class_id][1],
                "visible": True,
                "box": box if state == "label_with_box" else None,
            })
        source = demo_image_bytes(frame + texture_seed, marks)
        role = "observation"
        if task == "verified_reference_comparison":
            role = "target" if index == 0 else "verified_normal_reference"
        units.append({
            "unit_id": unit_id, "label": label, "video_id": video_id,
            "frame_index": frame, "source_sha256": hashlib.sha256(source).hexdigest(),
            "width": 96, "height": 64, "view": "panel" if multi else "original",
            "visible_chronology": sorted(v for v, _ in entries).index(frame) + 1 if multi else None,
            "unit_role": role, "observations": observations,
        })
    return {
        "evidence_origin": "synthetic_fixture",
        "profile_id": f"{task}.{profile}.v1",
        "recipe_id": f"demo-recipe-{task}-{profile}",
        "variant": variant, "evidence_state": state, "units": units,
        "scope_unit_ids": [unit["unit_id"] for unit in units],
        "display_scope_phrase": scope,
        "class_lexicon": {k: {"canonical": v[0], "public_surfaces": [v[0]], "aliases": [v[0]]}
                          for k, v in DEMO_CLASSES.items()},
        "groups_by_class": {k: [v[1]] for k, v in DEMO_CLASSES.items()},
        "query_class_id": query,
        "reference_unit_id": units[1]["unit_id"] if multi else None,
        "direction": "after", "normal_reference_class_ids": ["demo:normal"],
        "reference_criterion": "verified normal reference mucosa",
    }


def fixture_evidence_rows():
    for task in DEMO_PROFILES:
        yield task, synthetic_evidence(task)
        if task == "finding_identification":
            yield task, synthetic_evidence(task, landmark=True)


def offline_generate(public_request, attempt=1):
    """Emit a reproducible public wording for the test provider's one-based attempt."""
    if type(attempt) is not int or attempt < 1:
        raise ValueError("The simulated request number must be a positive integer")
    data = public_request.get("data", public_request)
    surfaces = {key.removesuffix("_surfaces"): values[0] for key, values in data.items()
                if key.endswith("_surfaces") and isinstance(values, list) and values}
    # These mock responses exercise two syntactic constructions for each profile.
    # Only public role values enter the text. The live provider has its own OpenAI adapter.
    examples = {
        "identify": (
            "Name one {kind} visible in {scope}.",
            "Could you give me the name of one {kind} you see in {scope}?",
        ),
        "classify_named_finding": (
            "In {scope}, which clinical group does the dataset assign to the {finding_term}?",
            "What is the dataset clinical group of the {finding_term} in {scope}?",
        ),
        "localize_evidence_frame": (
            "Which one of {display_set} provides visual evidence of a {finding}?",
            "Among {display_set}, identify one that shows a {finding}.",
        ),
        "localize": (
            "In {scope}, could you give {box_format} for a bounding box around one {finding_term}?",
            "Provide a bounding box for one {finding_term} in {scope} as {box_format}.",
        ),
        "select_reference_panel": (
            "Which panel in {comparison_scope} shows the {reference_criterion}?",
            "In {comparison_scope}, identify one panel that depicts the {reference_criterion}.",
        ),
        "select_earlier_observed_frame": (
            "Between {option_a} and {option_b}, which frame showing the {finding} was recorded earlier?",
            "Which frame showing the {finding} was acquired earlier, {option_a} or {option_b}?",
        ),
        "localize_relative_evidence": (
            "In {scope}, name one frame with evidence of a {finding} {direction} {reference}.",
            "Which frame in {scope} shows a {finding} {direction} {reference}?",
        ),
    }
    pair = examples[data["operation"]]
    return {"question": pair[(attempt - 1) % len(pair)].format(**surfaces)}


def materialize_synthetic_presentation(evidence, output_dir):
    """Write the synthetic input views and their public mapping, without an answer key."""
    if evidence.get("evidence_origin") != "synthetic_fixture":
        raise ValueError("The raster renderer is for synthetic examples only")
    destination = Path(output_dir)
    destination.mkdir(parents=True, exist_ok=True)
    public_units = []
    for index, unit in enumerate(evidence["units"]):
        # The drawing is deliberately schematic. Only the synthetic fixture specifies these marks.
        marks = [[12 + n * 40, 16, 36 + n * 40, 38]
                 for n, obs in enumerate(unit["observations"]) if obs["class_id"] != "demo:normal"]
        seed = int(hashlib.sha256(unit["video_id"].encode()).hexdigest()[:6], 16)
        data = demo_image_bytes(unit["frame_index"] + seed, marks)
        if hashlib.sha256(data).hexdigest() != unit["source_sha256"]:
            raise ValueError("Synthetic presentation does not match its evidence receipt")
        name = f"view_{index + 1}.pgm"
        (destination / name).write_bytes(data)
        public_units.append({"unit_id": unit["unit_id"], "label": unit["label"], "image": name,
                             "visible_chronology": unit["visible_chronology"]})
    presentation = {"origin": "synthetic_fixture", "scope": evidence["display_scope_phrase"],
                    "units": public_units}
    (destination / "presentation.json").write_text(
        json.dumps(presentation, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    return presentation


### 3 · Evidence, answers and references

These cases check the eight profiles, including scope, temporal direction, ambiguity and extra clinical requests. S8 reference tests exercise the existing evidence identity and presentation rules.


In [6]:
def run_fixture_regressions(contracts):
    """Execute independent expectations against the same adapters and comparator used by the run."""
    passed = []
    def expect(name, actual, gold):
        if actual != gold:
            raise AssertionError(f"{name}: expected {gold!r}, got {actual!r}")
        passed.append(name)

    cases = {}
    for task, record in fixture_evidence_rows():
        case = build_case(task, record, contracts)
        cases[case["profile_id"]] = case
        candidate = offline_generate(case["public_request"])
        prefix = case["profile_id"]
        verdict = validate_candidate(case, candidate)
        expect(prefix + " / wording", verdict["verdict"], "accepted")
        expect(prefix + " / extra clinical act", validate_candidate(
            case, {"question": candidate["question"] + " Recommend a treatment."}
        )["verdict"], "rejected")
        expect(prefix + " / extra JSON key", validate_candidate(
            case, dict(candidate, answer="private")
        )["verdict"], "rejected")
        # The text has a speculative register and no diagnosed contradiction. The
        # bounded parser cannot certify it; uncertainty cannot produce acceptance.
        expect(prefix + " / unsupported wording", validate_candidate(
            case, {"question": "Kindly consider this request: " + candidate["question"]}
        )["verdict"], "review")
        with tempfile.TemporaryDirectory() as tmp:
            presentation = materialize_synthetic_presentation(record, tmp)
            hashes = [hashlib.sha256((Path(tmp) / u["image"]).read_bytes()).hexdigest()
                      for u in presentation["units"]]
            expect(prefix + " / displayed bytes", hashes, [u["source_sha256"] for u in record["units"]])

    def demo_case(task, landmark=False):
        return build_case(task, synthetic_evidence(task, landmark=landmark), contracts)

    land = demo_case("finding_identification", True)
    for name, q, gold in [
        ("rank", "Which anatomical landmark is most obvious in image_1?", "rejected"),
        ("cardinality", "Name every anatomical landmark in image_1.", "rejected"),
        ("disclosure", "Is the anatomical landmark in image_1 the ampulla of Vater?", "rejected"),
        ("article", "Name a anatomical landmark in image_1.", "rejected"),
        ("novel scope", "Name one anatomical landmark visible in panel Z.", "rejected"),
        ("extra attribute", "Name one inflamed anatomical landmark in image_1.", "rejected"),
        ("eligible additional syntax", "What anatomical landmark do you see in image_1?", "accepted"),
        ("indirect identification", "Could you tell me which anatomical landmark you see in image_1?", "accepted"),
        ("indirect count", "Could you tell me how many anatomical landmarks you see in image_1?", "rejected"),
        ("indirect wrong scope", "Could you tell me which anatomical landmark you see in panel Z?", "rejected"),
        ("indirect extra request", "Could you tell me which anatomical landmark you see in image_1? Recommend a treatment.", "rejected"),
        ("indirect with unresolved continuation", "Could you tell me which anatomical landmark you see in image_1, if applicable?", "review"),
    ]:
        expect("identification / " + name, validate_candidate(land, {"question": q})["verdict"], gold)

    ambiguous = dict(land, ambiguous_public_referents=True)
    expect("ambiguous scope alone", validate_candidate(
        ambiguous, {"question": "Name one anatomical landmark in image_1."})["verdict"], "review")
    expect("ambiguous scope with certain conflict", validate_candidate(
        ambiguous, {"question": "Name every anatomical landmark in image_1."})["verdict"], "rejected")
    for task, expected_answers in {
        "evidence_frame_localization": ["image_1/panel_1", "image_1/panel_3"],
        "verified_reference_comparison": ["image_1/panel_2"],
        "observed_anchor_order": ["image_1/panel_2"],
        "temporal_relative_evidence_localization": ["image_1/panel_3", "image_1/panel_4"],
        "optional_spatial_localization": [[12,16,36,38], [52,16,76,38]],
        "finding_category": ["mucosal_lesion"],
    }.items():
        expect(task + " / physical reference", demo_case(task)["private_answer"]["alternatives"], expected_answers)

    temporal = demo_case("temporal_relative_evidence_localization")
    question = offline_generate(temporal["public_request"])["question"]
    expect("temporal / direction reversed", validate_candidate(
        temporal, {"question": question.replace("after", "before")})["verdict"], "rejected")
    ref = demo_case("verified_reference_comparison")
    expect("comparison / winner suggested", validate_candidate(ref, {
        "question": "Which panel in this comparison shows the verified normal reference mucosa, panel B?"
    })["verdict"], "rejected")

    def expect_hold(task, edit, name, disposition=None):
        record = synthetic_evidence(task)
        edit(record)
        try:
            build_case(task, record, contracts)
        except EvidenceHold as hold:
            if disposition is not None:
                expect(name, hold.disposition, disposition)
            else:
                passed.append(name)
            return
        raise AssertionError(f"{name}: evidence was admitted")
    expect_hold("finding_category",
                lambda r: r["groups_by_class"]["demo:polyp"].append("anatomical_landmark"),
                "ambiguous dataset group", "review")
    expect_hold("optional_spatial_localization",
                lambda r: r["units"][0]["observations"][0].update(box=[0, 0, 250, 100]),
                "out-of-bounds geometry", "review")
    expect_hold("observed_anchor_order",
                lambda r: r["units"][1].update(video_id="different-video"),
                "temporal comparison across videos", "review")
    expect_hold("evidence_frame_localization",
                lambda r: r["units"][1].update(label=r["units"][0]["label"]),
                "duplicate public panel label", "review")
    expect_hold("verified_reference_comparison",
                lambda r: r["units"][1].update(observations=[]),
                "missing annotation cannot establish normality", "review")
    expect_hold("temporal_relative_evidence_localization",
                lambda r: r.update(reference_unit_id=r["units"][-1]["unit_id"], direction="after"),
                "empty strict target side", "not_applicable")
    return {"synthetic_profiles": len(cases), "checks_passed": len(passed), "checks": passed}


def run_s8_reference_regressions():
    """Check direct reuse of S8 inputs without reading pixels or recomputing image hashes."""
    pin = {"repository": "example/mmvqa", "resolved_commit": "a" * 40, "source_catalog_sha256": "b" * 64}
    inputs, units = [], []
    for index, frame in enumerate((17, 24)):
        item = {"input_order": index, "relative_path": f"data/labelled_images/Polyp/{frame}.jpg",
                "storage": "original", "width": 336, "height": 336, "sha256": str(index+1) * 64,
                "input_id": str(index+3) * 64, "artifact_id": str(index+5) * 64,
                "image_role": "target" if index == 0 else "temporal_context"}
        inputs.append(item)
        units.append({"unit_id": f"image_{index+1}", "input_id": item["input_id"],
                      "artifact_id": item["artifact_id"], "frame_index": frame, "panel_index": None,
                      "source_sha256": item["sha256"], "operation": "full", "geometry": {},
                      "annotation_assisted": False})
    recipe = {"example_id": "c" * 64, "variant": "temporal_multi", "image_count": len(inputs),
              "pixels_available": True, "annotation_assisted": False, "ordered_inputs": inputs}

    checked = []
    def expect(name, condition):
        if not condition:
            raise AssertionError(name)
        checked.append(name)

    no_io = AssertionError("Direct S8 references must use the published metadata alone")
    with patch("builtins.open", side_effect=no_io), patch.object(Path, "open", side_effect=no_io), \
         patch.object(hashlib, "sha256", side_effect=no_io):
        ref = _s8_presentation_reference(pin, recipe, units)
    expect("Direct reference construction needs no image I/O or new image hash", bool(ref))
    expect("The S8 snapshot and example are pinned",
           all(ref[key] == pin[key] for key in pin) and ref["example_id"] == recipe["example_id"])
    keys = ("input_order", "input_id", "artifact_id", "relative_path", "storage", "sha256", "width", "height")
    expect("Every ordered input retains the S8 locator, digest and dimensions",
           [[i[k] for k in keys] for i in ref["ordered_inputs"]] == [[i[k] for k in keys] for i in inputs])
    expect("Each public unit resolves to the corresponding input",
           [[u["unit_id"] for u in i["units"]] for i in ref["ordered_inputs"]] == [["image_1"], ["image_2"]])

    # A direct S8 panel must retain its original on-image caption when screening disclosure.
    artifact_id = inputs[0]["artifact_id"]
    visual = {artifact_id: {"metadata": {"clinical_labels_rendered": False, "qc_scores_rendered": False}}}
    panel = dict(units[0], unit_id="image_1/panel_1", panel_index=0,
                 geometry={"label": "A", "panel_origin_xy": [0, 0], "image_region_xyxy": [12, 18, 324, 330]})
    expect("Neutral panel labels are eligible",
           _s8_display_review_reason(visual, [panel], "evidence_frame_localization") is None)
    flags_unknown = {artifact_id: {"metadata": {"qc_scores_rendered": False}}}
    expect("A composite with missing disclosure metadata is held",
           _s8_display_review_reason(flags_unknown, [panel], "finding_identification")
           == "review_unverified_S8_display_annotations")
    expect("Assisted overlays are held without rerendering",
           _s8_display_review_reason(visual, [dict(panel, operation="overlay")], "finding_identification")
           == "review_assisted_overlay")
    captioned = dict(panel, geometry=dict(panel["geometry"], label="Frame 17 +0.000s target"))
    expect("Target captions that could give away the selected frame go to review",
           _s8_display_review_reason(visual, [captioned], "evidence_frame_localization")
           == "review_S8_target_marker_may_disclose_answer")
    for field in ("relative_path", "sha256"):
        incomplete = dict(inputs[0])
        del incomplete[field]
        try:
            _s8_presentation_reference(pin, dict(recipe, ordered_inputs=[incomplete, inputs[1]]), units)
        except KeyError as exc:
            expect("A reference requires the published " + field, exc.args == (field,))
        else:
            raise AssertionError("Accepted an incomplete S8 locator: " + field)
    return {"checks_passed": len(checked), "checks": checked}


### 4 · V13 wording benchmark

Each executable case must match its expected final verdict and every required individual check. A missing result or mismatched status fails even if the final verdict matches. The report lists skipped review-only or unsupported contexts separately.


In [7]:
# These two category answers are transcriptions of *context assumptions*, never gold verdicts.
# The context digest prevents accidental reuse if a future contract changes the private assumptions.
_BENCHMARK_GROUP_CONTEXTS = {
    "ad48a3df3937b5bad1062c3a57c4fe8ce5c984fb16988f9e06f21359c48ed7f8": "mucosal_lesion",
    "2179660e6766ec2eb1d149ad50af1cb0aa3431134bd61af8c423ec860ba9642e": "foreign_body",
}


def _wording_fixture_inputs(task, pair, context):
    grammar, semantics = pair["grammar"], pair["semantics"]
    public_context = context["public_context"]
    if context.get("kind_approval_overlay_id"):
        # These fixtures test optional synthetic role-approval overlays. This implementation
        # uses fixed category labels; an overlay is never turned into an S1 approval.
        return None, "synthetic_kind_approval_overlay_outside_active_policy"
    if task == "finding_identification":
        roles = {"KIND": public_context["kind_surfaces_by_profile"][context["profile_id"]],
                 "SCOPE": public_context["scope_surfaces_by_id"][context["parameters"]["scope_id"]]}
        blocked = [s for alternative in context["private_test_reference"]["eligible_alternatives"]
                   for s in alternative["disclosure_surfaces"]]
    else:
        roles = public_context["role_surfaces"]
        blocked = []
    view = context["public_presentation"]
    units = view.get("displayed_units", [])
    # Preserve both fixture-internal and externally visible identifiers.
    unit_names = [[u[key] for key in ("unit_id", "label", "public_label", "visible_label") if key in u]
                  for u in units]
    if task == "finding_category":
        group = _BENCHMARK_GROUP_CONTEXTS.get(wire_digest(context))
        if group is None:
            return None, "unbound_private_group_in_changed_fixture_context"
        blocked = [group, group.replace("_", " ")]
    elif task == "evidence_frame_localization":
        blocked = [name for names in unit_names for name in names]
    elif task == "temporal_relative_evidence_localization":
        permitted = set(roles["REFERENCE"])
        blocked = [name for names in unit_names if permitted.isdisjoint(names) for name in names]
    elif task == "verified_reference_comparison":
        blocked = [label for group in view.get("displayed_comparisons", []) for label in group["panel_labels"]]
    # observed_anchor_order must permit both named operands. Spatial location checks screen
    # numeric coordinates and leave unsupported spatial language unresolved.
    request_schema = grammar["public_request"]["data_schema"]
    request = {key: spec["const"] for key, spec in request_schema["properties"].items() if "const" in spec}
    for key, binding in grammar["public_request"]["field_bindings"].items():
        if "resolved_slot" in binding:
            spec = binding["resolved_slot"]
            if spec["root"] != "grammar_document" or spec["path"][:-1] != ["slot_bindings"]:
                return None, "unimplemented_fixture_slot_binding"
            request[key] = roles[spec["path"][-1]]
    Draft202012Validator(request_schema).validate(request)
    ambiguous_group = (len(view.get("displayed_comparisons", [])) > 1
                       and not view.get("focused_comparison") and not view.get("pointing_gesture"))
    profile = next(p for p in semantics["profiles"] if p["profile_id"] == context["profile_id"])
    constants = profile["plan_constants"]
    operation = profile.get("public_operation") or semantics["task_semantics"]["public_operation"]
    # These IDs belong only to the offline presentation and are never sent to the generator.
    bound_roles = {slot: f"{task}/{context['profile_id']}/{slot}" for slot in roles}
    plan = {**constants, "task_id": task, "profile_id": context["profile_id"],
            "public_operation": operation, "roles": bound_roles}
    return {"task_id": task, "profile_id": context["profile_id"], "frozen_plan": plan,
            "role_surfaces": roles, "public_request": request, "disclosure_terms": blocked,
            "ambiguous_public_referents": ambiguous_group}, None


def _required_fixture_checks(assertions, actual_checks, registry):
    """Resolve each expected ID to exactly one semantic check result."""
    statuses = defaultdict(list)
    for check in actual_checks:
        statuses[check["check_id"]].append(check["status"])
    comparisons = []
    for assertion in assertions:
        check_id, expected = assertion["check_id"], assertion["verdict"]
        observed = statuses.get(check_id, [])
        if check_id not in registry:
            error = "unregistered_check"
        elif not observed:
            error = "missing_result"
        elif len(observed) != 1:
            error = "duplicate_result"
        elif observed[0] != expected:
            error = "status_mismatch"
        else:
            error = None
        comparisons.append({"check_id": check_id, "expected": expected,
                            "observed_statuses": observed, "error": error})
    return comparisons


def benchmark_contract_wording(contracts):
    """Compare the final verdict and every required check on each executable V13 fixture."""
    totals = Counter()
    rows = []
    for task, pair in sorted(contracts.items()):
        group = pair["grammar"]["language_fixtures"]
        registry = pair["semantics"]["check_registry"]
        if group["excluded_from_generation_request"] is not True:
            raise ValueError("unexpected visibility of held-out contract fixture")
        for fixture in group["cases"]:
            context = group["contexts"][fixture["context_id"]]
            case, skip = _wording_fixture_inputs(task, pair, context)
            expected = fixture["gold"]["final_verdict"]
            row = {"task": task, "fixture": fixture["id"], "context": fixture["context_id"],
                   "expected": expected}
            rows.append(row)
            totals["total"] += 1
            if skip:
                row.update(outcome="skipped", observed="not_run", reasons=[skip], assertions=[])
                totals["skipped"] += 1
                continue

            actual = validate_candidate(case, {"question": fixture["text"]})
            observed = actual["verdict"]
            comparisons = _required_fixture_checks(fixture.get("required_check_outcomes", []),
                                                   actual["checks"], registry)
            passed_checks = sum(item["error"] is None for item in comparisons)
            matched = observed == expected and passed_checks == len(comparisons)
            row.update(outcome="passed" if matched else "failed", observed=observed,
                       reasons=actual["reasons"], assertions=comparisons)
            totals["executed"] += 1
            totals["matched"] += int(matched)
            totals["assertions_evaluated"] += len(comparisons)
            totals["assertions_matched"] += passed_checks
            totals["false_accepts"] += int(observed == "accepted" and expected != "accepted")
            totals["false_rejects"] += int(observed == "rejected" and expected == "accepted")
            totals["abstentions_on_definitive_gold"] += int(observed == "review" and expected != "review")
            totals["expected_review"] += int(expected == "review")
    summary = {key: totals[key] for key in (
        "total", "executed", "skipped", "matched", "assertions_evaluated", "assertions_matched",
        "false_accepts", "false_rejects", "abstentions_on_definitive_gold", "expected_review"
    )}
    return {"summary": summary, "cases": rows}


def test_required_fixture_outcomes(contracts):
    """Reproduce false passes when the final verdict is right but a required check fails."""
    task = "finding_identification"
    group = contracts[task]["grammar"]["language_fixtures"]
    fixture = next(f for f in group["cases"] if f["id"] == "check_presence")
    pair = {**contracts[task], "grammar": {**contracts[task]["grammar"],
                                          "language_fixtures": {**group, "cases": [fixture]}}}
    case, _ = _wording_fixture_inputs(task, pair, group["contexts"][fixture["context_id"]])
    actual = validate_candidate(case, {"question": fixture["text"]})
    check_id = task + ".question_semantic_form"
    other_checks = [check for check in actual["checks"] if check["check_id"] != check_id]
    wrong_status = {"check_id": check_id, "status": "review"}
    for checks, error in ((other_checks, "missing_result"), (other_checks + [wrong_status], "status_mismatch")):
        with patch.dict(benchmark_contract_wording.__globals__, validate_candidate=Mock(return_value={**actual, "checks": checks})):
            report = benchmark_contract_wording({task: pair})
        row = report["cases"][0]
        if (row["expected"] != row["observed"] or report["summary"]["matched"] != 0
                or not any(item["check_id"] == check_id and item["error"] == error for item in row["assertions"])):
            raise AssertionError(f"The benchmark did not detect {error} for {check_id}")


### 5 · Source import, integrity and selective repair

An isolated S1 publication and a local GitHub fixture exercise V13 and Sector 1 file admission. Tests count SHA-256 evaluations for individual inputs and record which source files are opened during repairs.


In [8]:
def run_manifest_regressions(staged_directory):
    """Test the staging boundary using temporary copies of an already verified release.

    Network responses are simulated. Each transfer, write, read-back and digest check
    uses the same code as a real run. The caller's input directory is read-only here.
    """

    manifest_name = CONTRACT_MANIFEST_NAME
    manifest_repo_path = CONTRACT_PREFIX + "/" + manifest_name
    original = Path(staged_directory)
    manifest_bytes = (original / manifest_name).read_bytes()
    manifest = json.loads(manifest_bytes)
    names = sorted(row["path"] for row in manifest["files"])
    passed = []

    def digest(data):
        return hashlib.sha256(data).hexdigest()

    def encode(value):
        return (json.dumps(value, ensure_ascii=False, indent=2) + "\n").encode()

    def check(label, condition):
        if not condition:
            raise AssertionError(label)
        passed.append(label)

    def must_reject(label, action, expected_errors=(ValueError,)):
        try:
            action()
        except expected_errors:
            passed.append(label)
        else:
            raise AssertionError("The import unexpectedly succeeded: " + label)

    class RepositoryFixture:
        def __init__(self):
            self.head = hashlib.sha1(b"first test revision").hexdigest()
            self.snapshots = {self.head: {CONTRACT_PREFIX + "/" + n: (original / n).read_bytes()
                                         for n in [manifest_name, *names]}}
            self.head_reads = 0
            self.pinned_reads = []
            self.raw_reads = []
            self.override = {}

        def metadata(self, endpoint):
            if endpoint == "commits/main":
                self.head_reads += 1
                return {"sha": self.head}
            prefix = "contents/"
            if not endpoint.startswith(prefix):
                raise AssertionError("Unexpected GitHub endpoint: " + endpoint)
            path, ref = unquote(endpoint[len(prefix):]).rsplit("?ref=", 1)
            self.pinned_reads.append(ref)
            if path not in self.snapshots[ref]:
                raise HTTPError("https://example.test", 404, "Missing fixture file", {}, None)
            data = self.snapshots[ref][path]
            git_id = hashlib.sha1(b"blob " + str(len(data)).encode() + b"\0" + data).hexdigest()
            return {"path": path, "type": "file", "size": len(data), "sha": git_id}

        def read(self, commit, path):
            self.pinned_reads.append(commit)
            self.raw_reads.append(path)
            return self.override.get((commit, path), self.snapshots[commit][path])

        def open(self, cache, sidecar=None):
            return open_contract_release(cache, online=True, manifest_path=sidecar,
                                         github_json=self.metadata, read_bytes=self.read)

    with tempfile.TemporaryDirectory() as temp:
        root = Path(temp)
        repo = RepositoryFixture()
        cache = root / "cache"
        contract_hashes = [row["sha256"] for row in manifest["files"]]
        with count_file_hashes() as hashes:
            live = repo.open(cache)
        check("Importing V13 hashes each downloaded contract exactly once",
              all(hashes[h] == 1 for h in contract_hashes))
        verify_impl = verify_staged_contract_release
        directory = Path(live["staged_directory"])
        staged_manifest = directory / manifest_name

        def contract_downloads():
            return [path for path in repo.raw_reads if path != manifest_repo_path]

        def file_identity(name):
            stat = (directory / name).stat()
            return stat.st_ino, stat.st_mtime_ns

        check("A fresh VM downloads exactly the fourteen declared contracts",
              Counter(contract_downloads()) == Counter(CONTRACT_PREFIX + "/" + n for n in names))
        check("One main lookup and one immutable revision per import",
              repo.head_reads == 1 and set(repo.pinned_reads) == {repo.head} and live["registry_commit"] == repo.head)
        check("The verified stage can be reopened from its reported paths",
              staged_manifest == Path(live["staged_manifest_path"]) and staged_manifest.read_bytes() == manifest_bytes
              and len(verify_impl(directory, live["manifest_sha256"])["contracts"]) == 7)

        # A good local contract keeps both its file identity and its modification time.
        original_files = {n: file_identity(n) for n in names}
        repo.raw_reads.clear()
        with count_file_hashes() as hashes:
            reopened = repo.open(cache)
        check("A fresh import of healthy V13 files computes one local digest per contract",
              all(hashes[h] == 1 for h in contract_hashes))
        check("A warm VM reuses the verified directory without downloading or replacing any contract",
              Path(reopened["staged_directory"]) == directory and not contract_downloads()
              and original_files == {n: file_identity(n) for n in names})

        missing, corrupted = names[0], names[-1]
        (directory / missing).unlink()
        bad_data = (directory / corrupted).read_bytes()
        (directory / corrupted).write_bytes(bytes([bad_data[0] ^ 1]) + bad_data[1:])
        repo.raw_reads.clear()
        healed = repo.open(cache)
        check("Only a missing contract and a same-size damaged contract are fetched again",
              Counter(contract_downloads()) == Counter([CONTRACT_PREFIX + "/" + missing, CONTRACT_PREFIX + "/" + corrupted])
              and Path(healed["staged_directory"]) == directory
              and all((directory / n).read_bytes() == (original / n).read_bytes() for n in names)
              and all(file_identity(n) == original_files[n] for n in names if n not in (missing, corrupted)))

        # Partial imports remain resumable after a failed download.
        interrupted_cache = root / "interrupted-cache"
        last_contract = CONTRACT_PREFIX + "/" + names[-1]
        repo.raw_reads.clear()
        repo.override[repo.head, last_contract] = b"invalid-published-bytes"
        must_reject("A failed cold download prevents publication",
                    lambda: repo.open(interrupted_cache))
        transferred = set(contract_downloads()) - {last_contract}
        remaining = {CONTRACT_PREFIX + "/" + n for n in names} - transferred
        del repo.override[repo.head, last_contract]
        repo.raw_reads.clear()
        recovered = repo.open(interrupted_cache)
        check("An interrupted cold import keeps verified files on the VM",
              bool(transferred) and Counter(contract_downloads()) == Counter(remaining) and len(recovered["contracts"]) == 7)

        # Re-import a pinned release after local damage while repository main has advanced.
        damaged_path = CONTRACT_PREFIX + "/" + corrupted
        canonical = (original / corrupted).read_bytes()
        damaged = bytes([canonical[0] ^ 1]) + canonical[1:]
        (directory / corrupted).write_bytes(damaged)
        repo.override[repo.head, damaged_path] = damaged
        good_before = {n: file_identity(n) for n in names if n != corrupted}
        initial_revision = repo.head
        repo.head = hashlib.sha1(b"unrelated current main").hexdigest()
        repo.snapshots[repo.head] = {}
        repo.raw_reads.clear()
        repo.pinned_reads.clear()
        initial_head_reads = repo.head_reads
        reopen = lambda: open_contract_release(cache, online=True, pinned_commit=initial_revision,
                                               github_json=repo.metadata, read_bytes=repo.read)
        must_reject("An unrecoverable replacement prevents admission of the release",
                    reopen, (ValueError, RuntimeError))
        check("Failed repair targets only the pinned bad file",
              set(contract_downloads()) == {damaged_path} and len(contract_downloads()) == 3
              and repo.head_reads == initial_head_reads and set(repo.pinned_reads) == {initial_revision}
              and all(file_identity(n) == good_before[n] for n in good_before))
        del repo.override[initial_revision, damaged_path]
        repo.raw_reads.clear()
        reopen()
        check("A fresh import repairs one file using the original commit",
              contract_downloads() == [damaged_path] and (directory / corrupted).read_bytes() == canonical
              and all(file_identity(n) == good_before[n] for n in good_before))
        full_identity = {n: file_identity(n) for n in names}
        staged_manifest.write_bytes(manifest_bytes + b" ")
        repo.raw_reads.clear()
        reopen()
        check("Import restores the authenticated manifest without downloading healthy contracts",
              staged_manifest.read_bytes() == manifest_bytes and not contract_downloads()
              and all(file_identity(n) == full_identity[n] for n in names))
        repo.head = initial_revision

        name = names[-1]
        path = directory / name
        good_bytes = path.read_bytes()
        # Edit one VM byte without changing the length or the network response.
        path.write_bytes(bytes([good_bytes[0] ^ 1]) + good_bytes[1:])
        must_reject("Same-size change to a previously staged file fails read-back verification",
                    lambda: verify_impl(directory, live["manifest_sha256"]))
        path.write_bytes(good_bytes)
        staged_manifest.write_bytes(manifest_bytes + b" ")
        must_reject("Substituting a valid JSON manifest on disk breaks its trusted digest",
                    lambda: verify_impl(directory, live["manifest_sha256"]))
        staged_manifest.write_bytes(manifest_bytes)
        path.unlink()
        must_reject("A missing file stops verification", lambda: verify_impl(directory, live["manifest_sha256"]),
                    (ValueError, FileNotFoundError))
        path.write_bytes(good_bytes)

        # The first malformed contract now matches this *test* manifest. The last
        # contract has a digest mismatch. A parser invoked too early would raise a
        # JSON error for the first file instead of reporting the later digest error.
        poison = root / "poison"
        shutil.copytree(directory, poison)
        first = names[0]
        invalid_text = b"this file is deliberately not JSON"
        (poison / first).write_bytes(invalid_text)
        (poison / names[-1]).write_bytes(good_bytes + b" ")
        revised = copy.deepcopy(manifest)
        for entry in revised["files"]:
            if entry["path"] == first:
                entry.update(sha256=digest(invalid_text), bytes=len(invalid_text))
        changed_manifest = encode(revised)
        (poison / manifest_name).write_bytes(changed_manifest)
        try:
            verify_impl(poison, digest(changed_manifest))
        except ValueError as exc:
            check("The last bad digest is reported before the first contract can be parsed",
                  not isinstance(exc, json.JSONDecodeError) and names[-1] in str(exc))
        else:
            raise AssertionError("Unverified contract reached the parser")
        (poison / first).write_bytes((original / first).read_bytes())
        (poison / names[-1]).write_bytes(good_bytes)

        def bad_manifest(label, content):
            (poison / manifest_name).write_bytes(content)
            must_reject(label, lambda: verify_impl(poison, digest(content)))

        missing = copy.deepcopy(manifest)
        missing["files"].pop()
        bad_manifest("A missing declaration is rejected", encode(missing))
        repeated = copy.deepcopy(manifest)
        repeated["files"].append(repeated["files"][0])
        bad_manifest("A repeated contract name is rejected", encode(repeated))
        unexpected = copy.deepcopy(manifest)
        unexpected["files"][0]["path"] = "unknown_task.grammar.v13.json"
        bad_manifest("An unexpected task file is rejected", encode(unexpected))
        bad_sha = copy.deepcopy(manifest)
        bad_sha["files"][0]["sha256"] = "invalid"
        bad_manifest("A malformed SHA-256 is rejected", encode(bad_sha))
        bad_size = copy.deepcopy(manifest)
        bad_size["files"][0]["bytes"] += 1
        bad_manifest("A mismatched manifest byte count is rejected", encode(bad_size))
        bad_manifest("Malformed manifest JSON is rejected", b"{bad")
        bad_manifest("Repeated manifest JSON keys are rejected", b'{"algorithm":"sha256",' + encode(manifest)[1:])

        prior = repo.head
        repo.head = hashlib.sha1(b"next test revision").hexdigest()
        files = repo.snapshots[repo.head] = dict(repo.snapshots[prior])
        del files[manifest_repo_path]
        sidecar = root / manifest_name
        sidecar.write_bytes(manifest_bytes)
        attached = repo.open(root / "cache", sidecar)
        check("A detached manifest verifies the current revision when GitHub has no manifest",
              attached["registry_commit"] == repo.head and attached["manifest_sha256"] == digest(manifest_bytes))
        contract_path = CONTRACT_PREFIX + "/" + name
        repo.head = hashlib.sha1(b"changed files in a distinct revision").hexdigest()
        files = repo.snapshots[repo.head] = dict(files)
        files[contract_path] += b"\n"
        must_reject("A changed remote contract fails validation against an outdated detached manifest on a fresh VM",
                    lambda: repo.open(root / "fresh-cache", sidecar))
        newer_manifest = copy.deepcopy(manifest)
        for entry in newer_manifest["files"]:
            if entry["path"] == name:
                entry.update(sha256=digest(files[contract_path]), bytes=len(files[contract_path]))
        files[manifest_repo_path] = encode(newer_manifest)
        repo.head_reads, repo.pinned_reads = 0, []
        newer = repo.open(root / "cache")
        check("Each online run follows the current main commit and its new published manifest",
              repo.head_reads == 1 and set(repo.pinned_reads) == {repo.head} and
              newer["contract_sha256"][name] == digest(files[contract_path]))
        must_reject("A stale detached manifest cannot override the published manifest",
                    lambda: repo.open(root / "cache", sidecar))

        # Independently verify the executable reader that the real-data path loads.
        reader_path = "notebooks/phase3/phase3_01_terminology_mapping.ipynb"
        data = files[reader_path] = b'{"nbformat":4}'
        def read_reader(cache):
            return read_github_snapshot_file(repo.head, reader_path, cache,
                                             github_json=repo.metadata, read_bytes=repo.read)
        raw, receipt = read_reader(root / "reader")
        check("The runtime reader digest identifies the verified Git blob",
              raw == data and receipt["sha256"] == digest(data) and receipt["bytes"] == len(data))
        repo.override[repo.head, reader_path] = b'{"nbformat":5}'
        must_reject("A modified runtime reader is rejected", lambda: read_reader(root / "reader2"))

        # The campaign pins its first revision, even if main moves between Colab sessions.
        tracked = RepositoryFixture()
        old_head = tracked.head
        del tracked.snapshots[old_head][manifest_repo_path]
        sidecar = root / "initial-contracts.json"
        sidecar.write_bytes(manifest_bytes)
        with patch.dict(open_contract_release.__globals__, {"_github_metadata": tracked.metadata, "_raw_at_commit": tracked.read}):
            first = open_campaign_release(root/"first", root/"campaign", manifest_path=sidecar)
            tracked.head = hashlib.sha1(b"later revision on main").hexdigest()
            tracked.snapshots[tracked.head] = {**tracked.snapshots[old_head], manifest_repo_path: manifest_bytes}
            sidecar.unlink()
            resumed = open_campaign_release(root/"second", root/"campaign")
            check("Resume reuses the saved source revision and the detached manifest on another VM",
                  tracked.head_reads == 1 and first["registry_commit"] == resumed["registry_commit"] == old_head
                  and first["manifest_sha256"] == resumed["manifest_sha256"])
            refreshed = open_campaign_release(root/"third", root/"campaign", refresh=True)
            check("An explicit source refresh captures current main",
                  tracked.head_reads == 2 and refreshed["registry_commit"] == tracked.head)
        # Damage to the campaign's convenience copy must repair only that manifest.
        for kind in ("github", "detached"):
            mirrored = RepositoryFixture()
            auto_cache, campaign = root / ("vm-" + kind), root / ("campaign-" + kind)
            explicit = root / ("supplied-" + kind + ".json")
            explicit.write_bytes(manifest_bytes)
            if kind == "detached":
                del mirrored.snapshots[mirrored.head][manifest_repo_path]
            with patch.dict(open_campaign_release.__globals__, {
                    "_github_metadata": mirrored.metadata, "_raw_at_commit": mirrored.read}):
                with count_file_hashes() as hashes:
                    captured = open_campaign_release(auto_cache, campaign, manifest_path=explicit)
                expected = digest(manifest_bytes)
                check(f"The {kind} manifest SHA-256 is evaluated once on initial import", hashes[expected] == 1)
                backup = campaign / "source" / (expected + ".json")
                for damage in ("missing", "corrupt"):
                    if damage == "missing":
                        backup.unlink()
                    else:
                        backup.write_bytes(b"damaged durable manifest")
                    mirrored.raw_reads.clear()
                    repaired = open_campaign_release(auto_cache, campaign)
                    check(f"A healthy {kind} VM manifest repairs a {damage} saved copy in place",
                          repaired["manifest_sha256"] == expected and backup.read_bytes() == manifest_bytes
                          and mirrored.raw_reads == [])
                explicit.write_bytes(b"contradictory explicit sidecar")
                must_reject(f"A contradictory explicit {kind} sidecar cannot override the pinned manifest",
                            lambda: open_campaign_release(auto_cache, campaign, manifest_path=explicit))
                if kind == "github":
                    backup.unlink()
                    Path(repaired["staged_manifest_path"]).unlink()
                    mirrored.raw_reads.clear()
                    current = open_campaign_release(auto_cache, campaign)
                    check("When both Git-backed manifest copies are missing only the pinned manifest is fetched",
                          mirrored.raw_reads == [manifest_repo_path] and current["manifest_sha256"] == expected
                          and backup.read_bytes() == manifest_bytes)
                else:
                    backup.write_bytes(b"wrong")
                    Path(repaired["staged_manifest_path"]).unlink()
                    must_reject("When neither detached copy verifies a fresh import stops",
                                lambda: open_campaign_release(auto_cache, campaign), (RuntimeError, ValueError))

    return {"checks_passed": len(passed), "checks": passed}


def run_s1_import_regressions(registry):
    """Import a small but complete S1 publication through the production loader."""
    checks = []
    calls = []
    def expect(name, condition):
        if not condition:
            raise AssertionError(name)
        checks.append(name)

    run_id = "20300101T120000_000000Z"
    commit = "c" * 40
    root = f"outputs/phase3/sector1/metadata/{run_id}"
    contracts = registry["contracts"]
    required = {name for pair in contracts.values()
                for name in pair["semantics"]["source_binding"]["required_artifacts"]}
    required.update({"snomed_subset_concepts", "snomed_subset_terms", "phase2_source_snapshot",
                     "sector2_binding_contract"})
    payloads = {name + ".json": {"fixture": name} for name in required if name != "task_contract"}
    rules, matrix = [], []
    for task, pair in contracts.items():
        rule_id = pair["semantics"]["source_binding"]["rule_id"]
        rules.append({"task": task, "rule_id": rule_id})
        matrix.append({"task": task, "variant": "test", "evidence_state": "observed",
                       "original_policy_code": "A", "expected_status": "applicable", "rule_id": rule_id})
    payloads.update({
        "coverage_contract.json": matrix,
        "task_contract_enriched.json": {"TASK_SPECIFICATION": rules},
        "sector2_binding_contract.json": {
            "module": "sector2_binding_validator.py", "loader": "load_sector1_binding_contract",
            "required_artifacts": ["coverage_contract.json", "fixture_only.json"]},
        "fixture_only.json": {"fixture": "only the binder requires this input"},
    })
    encode = lambda obj: (json.dumps(obj, sort_keys=True) + "\n").encode("utf-8")
    remote = {f"{root}/results/{name}": encode(obj) for name, obj in payloads.items()}
    remote[f"{root}/results/sector2_binding_validator.py"] = b"# Synthetic S1 binder bytes.\n"
    def digest(content):
        return hashlib.sha256(content).hexdigest()
    manifest = []
    for path, data in remote.items():
        manifest.append({"relative_path": path.rsplit("/", 1)[1], "sha256": digest(data),
                         "bytes": len(data), "repository_path": path})
    latest = {"run_id": run_id, "layout_root": root, "root": root, "repository": MMVQA_REPO,
              "data_commit": commit, "integrity_status": "remote_bytes_sha256_verified",
              "required_sector2_artifacts_published": True}
    def add_manifest(name, obj):
        path = root + "/" + name + ".json"
        data = encode(obj)
        remote[path] = data
        latest[name + "_path"] = path
        latest[name + "_sha256"] = digest(data)
    add_manifest("files_manifest", [{k: v for k, v in row.items() if k != "repository_path"} for row in manifest])
    add_manifest("drive_artifacts_manifest", {"fixture": "published drive manifest"})
    add_manifest("publication_manifest", {**latest, "files": manifest})
    remote[LATEST_PATH] = encode(latest)
    def read(revision, path):
        if revision != commit:
            raise AssertionError("Read outside the synthetic release revision")
        calls.append(path)
        return remote[path]
    def metadata(endpoint):
        encoded_path, ref = endpoint.removeprefix("contents/").rsplit("?ref=", 1)
        path = unquote(encoded_path)
        if ref != commit:
            raise AssertionError("Metadata request outside the pinned revision")
        data = remote[path]
        blob = hashlib.sha1(f"blob {len(data)}\0".encode() + data).hexdigest()
        return {"type": "file", "path": path, "size": len(data), "sha": blob}
    release = {**registry, "registry_commit": commit, "_read_bytes": read, "_github_json": metadata}
    coverage = next(row for row in manifest if row["relative_path"] == "coverage_contract.json")
    digest_set = {row["sha256"] for row in manifest}
    with tempfile.TemporaryDirectory(prefix="mmvqa-s1-import-") as cache:
        with count_file_hashes() as hashes:
            source = load_verified_sector1(release, cache, progress=lambda *args: None)
        expect("The seven tasks use the authoritative applicable matrix cells",
               set(source["selected_cells"]) == set(contracts)
               and all(len(rows) == 1 for rows in source["selected_cells"].values()))
        expect("Overlapping S1 task and binder inputs receive a single digest per physical file",
               all(hashes[digest] == 1 for digest in digest_set))
        calls.clear()
        with count_file_hashes() as hashes:
            warm = load_verified_sector1(release, cache, progress=lambda *args: None)
        expect("Re-importing a warm S1 publication reads each published payload once with zero downloads",
               not calls and all(hashes[digest] == 1 for digest in digest_set))
        expected = remote[coverage["repository_path"]]
        paths = [p for p in Path(cache).rglob("*") if p.is_file() and p.stat().st_size == len(expected)
                 and p.read_bytes() == expected]
        expect("V13 rule selection and the binder share one admitted copy of the S1 matrix", len(paths) == 1)
        paths[0].write_bytes(b"!" * len(expected))
        calls.clear()
        fixed = load_verified_sector1(release, cache, progress=lambda *args: None)
        expect("Repairing the S1 matrix fetches only its published path",
               calls == [coverage["repository_path"]] and fixed["selected_cells"] == warm["selected_cells"])
    return {"checks_passed": len(checks), "checks": checks}


### 6 · Audit and provider isolation

The audit tests verify exact question and request binding, retry limits, duplicate handling, corpus release and synthetic isolation. A transport-level test submits a public question payload to an in-memory OpenAI Responses API double.


In [9]:
def run_audit_regressions(registry):
    """Isolated synthetic design tests, including simulated release decisions. No HTTP or API keys."""
    contracts = registry["contracts"]
    passed = []
    def expect(name, result, expected):
        if result != expected:
            raise AssertionError(f"{name}: got {result!r}, expected {expected!r}")
        passed.append(name)

    def must_fail(name, error, action):
        try:
            action()
        except error:
            passed.append(name)
        else:
            raise AssertionError("Missing integrity/permission rejection: " + name)

    task = "finding_identification"
    raw = synthetic_evidence(task, landmark=True)
    raw["respondent_presentation"] = {
        "source": "phase2_sector8", "repository": "test/project", "resolved_commit": "1" * 40,
        "source_catalog_sha256": "2" * 64, "example_id": "test-recipe",
        "ordered_inputs": [{"input_order": 0, "artifact_id": "artifact", "input_id": "input",
                            "sha256": "3" * 64, "width": 96, "height": 64,
                            "relative_path": "private/test-class/image.jpg", "storage": "original", "units": []}],
    }
    case = build_case(task, raw, contracts)
    words = offline_generate(case["public_request"])["question"]
    second = words.replace("Name one", "Please identify one")
    public = openai_payload(contracts[task]["grammar"], case["public_request"])
    source = {"run_id": "simulated-test-release", "qa_generation_authorized": False,
              "corpus_export_authorized": False}
    auth = generation_policy(source, "real_openai")
    expect("Both generation authorities recorded",
           (auth["sector1_qa_generation_authorized"], auth["effective_qa_generation_authorized"],
            auth["sector1_export_authorized"]), (False, True, False))

    with tempfile.TemporaryDirectory() as tmp:
        store = AuditStore(tmp)
        try:
            receipt = {"build_id": text_sha("isolated-test-build"), "kind": "synthetic_demo",
                       "authorization": generation_policy(None, "demo_offline"), "source_run_id": None,
                       "semantic_instance_id": case["semantic_instance_id"],
                       "evidence_snapshot_id": store.put("snapshots", raw),
                       "case_sha256": store.put("plans", case), "public_request_sha256": store.put("requests", public),
                       "parser_implementation_sha256": text_sha("isolated-validation-component"),
                       "presentation": {"kind": "synthetic_test_presentation"}}
            def batch(events, target, current_receipt):
                calls = []
                def produce(payload, digest, attempt):
                    if attempt != len(calls) + 1 or wire_digest(payload) != digest:
                        raise AssertionError("altered or repeated attempt input")
                    calls.append(attempt)
                    item = events[attempt-1]
                    if isinstance(item, Exception):
                        raise item
                    return {"question": item}, {"provider": "isolated_test_double"}
                out = sample_case(case, contracts[task]["grammar"], contracts[task]["semantics"],
                                  current_receipt, store, produce, target, len(events))
                return out, len(calls)

            blocked, calls = batch([words, SamplingProviderError("test_provider_unavailable"), second],
                                   2, receipt)
            expect("Fatal abort withholds earlier accepted wording",
                   (blocked["stop_reason"], blocked["accepted_count"], blocked["accepted"], calls),
                   ("blocked", 0, [], 2))
            expect("Fatal abort keeps real history", [a["disposition"] for a in blocked["attempts"]],
                   ["accepted", "fatal_abort"])
            out, calls = batch([words + " Recommend a treatment.", "Kindly consider: " + words, words,
                                words.swapcase(), second], 2, receipt)
            expect("Rejection, review and normalized duplicate consume one attempt each",
                   [a["disposition"] for a in out["attempts"]],
                   ["rejected", "review", "accepted", "duplicate", "accepted"])
            expect("Distinct first originals survive", [x["question"] for x in out["accepted"]], [words, second])
            expect("Bounded target stopping", (calls, out["stop_reason"], out["shortfall"]), (5, "target_reached", 0))
            partial, _ = batch([words, words], 2, receipt)
            expect("Budget exhaustion reports shortfall", (partial["stop_reason"], partial["shortfall"]),
                   ("attempt_budget_exhausted", 1))
            record = out["accepted"][0]
            def assemble(candidate, *, synthetic=True, src=None, corpus=False):
                return assemble_validated(candidate, store, synthetic=synthetic, source=src,
                                          build_id=receipt["build_id"], for_corpus=corpus)
            qa = assemble(record)
            expect("Exact private answer and candidate hash", (qa["answer"], qa["candidate_question_sha256"]),
                   (case["private_answer"]["canonical"], text_sha(words)))
            expect("Corpus index deduplicates reruns", (store.insert([qa]), store.insert([qa])), (1, 0))
            mutated = dict(record, question=words.replace("one", "one abnormal"))
            must_fail("Export detects modified question", ValueError, lambda: assemble(mutated))
            path = store.root / "private" / "pre" / (out["pre_evidence_receipt_id"] + ".json")
            original = path.read_bytes()
            try:
                path.write_bytes(original + b" ")
                must_fail("Modified receipt bytes are rejected", ValueError, lambda: assemble(record))
            finally:
                path.write_bytes(original)
            must_fail("Synthetic cannot enter training corpus", PermissionError,
                      lambda: assemble(record, corpus=True))
            # Simulate the decision layer only, with private files in a disposable directory.
            sim_receipt = dict(receipt, kind="verified_sector1", authorization=auth, source_run_id=source["run_id"],
                               presentation=raw["respondent_presentation"])
            simulated, _ = batch([words], 1, sim_receipt)
            simulation = simulated["accepted"][0]
            draft = assemble(simulation, synthetic=False, src=source)
            expect("Generation decision permits a local draft", draft["release_status"], "local_validated_draft")
            expect("Published S8 digest reaches the QA record",
                   draft["presentation"]["ordered_inputs"][0]["sha256"], "3" * 64)
            expect("Answer-bearing S8 locations stay private",
                   "relative_path" in draft["presentation"]["ordered_inputs"][0], False)
            altered = copy.deepcopy(sim_receipt)
            altered["presentation"]["ordered_inputs"][0]["sha256"] = "4" * 64
            wrong_reference, _ = batch([words], 1, altered)
            must_fail("Wrong S8 reference cannot be assembled", ValueError,
                      lambda: assemble(wrong_reference["accepted"][0], synthetic=False, src=source))
            must_fail("Generation authority cannot grant export", PermissionError,
                      lambda: assemble(simulation, synthetic=False, src=source, corpus=True))
            must_fail("Another source run cannot release the draft", PermissionError,
                      lambda: assemble(simulation, synthetic=False, src=dict(source, run_id="different-run")))
        finally:
            store.db.close()
        # The real entry point must reject a synthetic mode before touching upstream or a provider.
        for mode in ("demo_offline", "demo_openai"):
            guard = Mock(side_effect=AssertionError("Invalid mode invoked production I/O"))
            with patch.dict(execute_sector2.__globals__, OpenAIQuestionClient=guard, load_verified_sector1=guard):
                must_fail("Production excludes " + mode, ValueError,
                          lambda: execute_sector2({"mode": mode}, registry, {}))
            expect(mode + " produces zero external calls", guard.call_count, 0)
    return {"checks_passed": len(passed), "checks": passed}


In [10]:
def test_offline_provider(contracts):
    task = "finding_identification"
    case = build_case(task, synthetic_evidence(task, landmark=True), contracts)
    public = openai_payload(contracts[task]["grammar"], case["public_request"])
    candidate = offline_generate(case["public_request"])
    data = {"status": "completed", "id": "fixture-response", "output": [
        {"type": "message", "role": "assistant", "content": [
            {"type": "output_text", "text": json.dumps(candidate)}
        ]}
    ]}
    response = BytesIO(json.dumps(data).encode())
    response.headers = {"x-request-id": "fixture-request"}
    transport = Mock()
    transport.open.return_value = response
    with patch.dict(OpenAIQuestionClient.__init__.__globals__, build_opener=lambda *args: transport):
        client = OpenAIQuestionClient(1, key_getter=lambda: "offline-placeholder")
        question, metadata = client.generate(public, wire_digest(public))
    request = transport.open.call_args.args[0]
    if (transport.open.call_count != 1 or request.data != _wire_bytes(public)
            or question != candidate or metadata["request_id"] != "fixture-request"):
        raise AssertionError("The production adapter did not round-trip the exact public request and simulated response.")
    if API_MODEL != "gpt-6-luna" or json.loads(request.data)["model"] != "gpt-6-luna":
        raise AssertionError("The serialized Responses API request must target gpt-6-luna.")
    if client.reviewer_model != "gpt-6-luna":
        raise AssertionError("The API reviewer default must use the configured model.")
    if validate_candidate(case, question)["verdict"] != "accepted":
        raise AssertionError("The validator did not accept the mocked provider's known-good candidate.")
    return {"checks_passed": 3, "checks": ["OpenAI request and response round-trip in memory",
                                          "Serialized generation request uses gpt-6-luna",
                                          "Reviewer uses the same configured model"]}


### 7 · Inventory, budget and checkpoint tests

The suite runs the full Section 1 import path independently in `inventory_only` and `real_openai` using local test doubles for the V13, S1 and S8 sources. It also tests reusing an inventory for generation. Synthetic inputs drive the real census and batch entry points through transactions, retries, interrupted submissions, recovery and durable API budgets. All paid requests in these scenarios use injected test doubles.


In [11]:
def run_inventory_batch_regressions(registry):
    """Exercise inventory transactions and paid-work resumption entirely on isolated synthetic cases."""
    checks = []
    def expect(name, condition):
        if not condition:
            raise AssertionError(name)
        checks.append(name)

    contracts = registry["contracts"]
    source = {"run_id": "simulated-resume-test", "qa_generation_authorized": False,
              "corpus_export_authorized": False, "selected_cells": {}}
    auth = generation_policy(source, "real_openai")
    contracts = effective_contracts(contracts, auth)
    rows = []
    for task, raw in fixture_evidence_rows():
        recipe_id = raw["recipe_id"]
        raw["s1_bindings"] = [{"binding_id": "test-binding"}]
        raw["respondent_presentation"] = {
            "source": "phase2_sector8", "repository": "test/fixtures", "resolved_commit": "1" * 40,
            "source_catalog_sha256": "2" * 64, "example_id": recipe_id,
            "ordered_inputs": [{"input_order": 0, "input_id": "test-input", "artifact_id": "test-artifact",
                                "width": 96, "height": 64, "units": [], "storage": "original",
                                "relative_path": "private/fixture/image.jpg", "sha256": "3" * 64}]}
        matrix = {"task": task, "variant": raw["variant"], "evidence_state": raw["evidence_state"],
                  "rule_id": contracts[task]["semantics"]["source_binding"]["rule_id"],
                  "original_policy_code": "A", "expected_status": "applicable"}
        cells = source["selected_cells"].setdefault(task, [])
        if matrix not in cells:
            cells.append(matrix)
        rows.append((task, raw))

    first_task, first_raw = rows[0]
    source["selected_cells"][first_task].append(dict(
        source["selected_cells"][first_task][0], evidence_state="synthetic_inapplicable_state",
        original_policy_code="N", expected_status="not_applicable"
    ))
    evaluations = Counter()
    interrupted = True
    def recipe_stream(source_data, cache, tasks, *, corpus=None):
        def entries():
            for index, (task, raw) in enumerate(rows):
                name = raw["recipe_id"]
                def evaluate(stats, on_display_hold, task=task, raw=raw, index=index, name=name):
                    evaluations[name] += 1
                    yield task, raw
                    if index == 1 and interrupted:
                        raise KeyboardInterrupt("isolated interruption after one uncommitted case")
                    if index == 0:
                        yield task, raw
                        yield task, dict(raw, evidence_state="synthetic_inapplicable_state")
                        on_display_hold({"recipe": name, "variant": raw["variant"], "task": task,
                                         "stage": "before_generation", "disposition": "review",
                                         "reason": "synthetic_display_hold"})
                yield {"example_id": name, "variant": raw["variant"]}, evaluate
        return len(rows), entries()

    with tempfile.TemporaryDirectory() as tmp:
        root = Path(tmp)
        shared = root / "build"
        shared.mkdir()
        build_id = text_sha("isolated-full-census-and-resume-test")
        def census(db):
            with patch.dict(inventory_census.__globals__, _verified_recipe_stream=recipe_stream):
                return inventory_census(db, source, root / "s8", contracts, checkpoint_recipes=1)
        db = open_inventory(shared, build_id)
        try:
            try:
                census(db)
            except KeyboardInterrupt:
                progress = inventory_summary(db, contracts, 2)
                expect("Interrupted inventory reports partial counts and withholds the full estimate",
                       progress["recipes_examined"] == 1 and progress["target_accepted_questions"] is None)
            else:
                raise AssertionError("The simulated census interruption was suppressed")
        finally:
            db.close()
        interrupted = False
        db = open_inventory(shared, build_id)
        try:
            census(db)
            expect("The census skips committed recipes and replays an interrupted recipe",
                   evaluations[rows[0][1]["recipe_id"]] == 1 and evaluations[rows[1][1]["recipe_id"]] == 2)
            counts = dict(db.execute("SELECT task, COUNT(*) FROM inventory_instances GROUP BY task"))
            expected_counts = Counter(task for task, _ in rows)
            expect("All seven tasks are counted, including both identification profiles",
                   len(counts) == 7 and counts == dict(expected_counts))
            expect("Overlapping candidates count once", sum(counts.values()) == len(rows))
            previous = evaluations.copy()
            census(db)
            expect("A completed census never reconstructs its eligible instances", evaluations == previous)
            for (instance_id,) in db.execute("SELECT instance_id FROM inventory_instances"):
                case, evidence, matrix = load_inventory_instance(db, instance_id)
                if (case["semantic_instance_id"] != instance_id or matrix["original_policy_code"] != "A"
                        or case["evidence_snapshot_id"] != wire_digest(evidence)):
                    raise AssertionError("Broken frozen evidence or matrix link")
            checks.append("All stored plans retain their source evidence and A-cell identity")
            estimate = inventory_summary(db, contracts, 2)
            expect("An exact target is reported only after full inventory",
                   estimate["complete"] and estimate["target_accepted_questions"] == 2 * len(rows))
            reasons = {json.loads(record)["reason"] for (record,) in db.execute("SELECT record FROM inventory_holds")}
            expect("Non-applicable and display holds are traceable before API dispatch",
                   {"synthetic_display_hold", "matrix_cell_not_eligible"} <= reasons)

            estimate_run = shared / "runs" / "estimate"
            estimate_run.mkdir(parents=True)
            metadata = {"build_id": build_id, "source": {"run_id": source["run_id"]},
                        "authorization": generation_policy(source, "inventory_only"), "http_calls": 0}
            no_api = Mock(side_effect=AssertionError("The inventory-only path invoked generation or requested a key"))
            already_imported = Mock(side_effect=AssertionError("The census repeated the Section 1 import"))
            metadata["s8_vm_sync"] = {"metadata": {}, "visuals": {}}
            with patch.dict(run_real_census_and_generation.__globals__, prepare_real_s8_inputs=already_imported,
                            _clinical_s8_corpus=already_imported, generate_inventory_batch=no_api, _colab_key=no_api):
                run_real_census_and_generation(
                    {"mode": "inventory_only", "requested_count": 2, "s8_dir": root, "checkpoint_recipes": 1},
                    dict(source, verified_references={}, verified_digests={}), contracts, shared,
                    estimate_run, metadata, {"validation": text_sha("test-parser")},
                    s8_context=({}, {}, {}),
                )
            expect("Reopening the completed census consumes Section 1's import without rebuilding the corpus",
                   already_imported.call_count == 0)
            expect("Inventory-only mode writes a full estimate and requires no API key",
                   no_api.call_count == 0 and metadata["http_calls"] == 0
                   and json.loads((estimate_run / "inventory_summary.json").read_text())["complete"])

            payload_counts = Counter()
            def client_factory(limit, *, punctuation_retry=False):
                client = Mock(max_http_calls=limit, calls=0)
                def generate(payload, expected_sha256):
                    if client.calls >= limit or wire_digest(payload) != expected_sha256:
                        raise AssertionError("Invalid dispatch or budget exceeded")
                    client.calls += 1
                    payload_counts[expected_sha256] += 1
                    wording = 1 if punctuation_retry or payload_counts[expected_sha256] == 1 else 2
                    result = offline_generate(json.loads(payload["input"]), attempt=wording)
                    if punctuation_retry:
                        # Inject one cosmetic repeat to verify deduplication across a resumed job.
                        original = result["question"]
                        result["question"] = original.rstrip(".?") + ("." if original.endswith("?") else "?")
                    return result, {"provider": "isolated_test_double"}
                client.generate.side_effect = generate
                return client

            config = {"model": API_MODEL, "requested_count": 2, "attempt_budget": 4,
                      "batch_size": 1, "max_batches_per_run": 1, "max_http_calls": 1}
            def generate_batch(name, factory=client_factory):
                run = shared / "runs" / name
                run.mkdir(parents=True)
                info = {"source": {"run_id": source["run_id"]}}
                generate_inventory_batch(config, db, contracts, source, auth, build_id,
                                         {"validation": text_sha("test-parser")}, run, info,
                                         client_factory=factory)
                return info, [json.loads(line) for line in
                              (run / "accepted_questions.jsonl").read_text().splitlines()]

            first, drafts1 = generate_batch("first")
            expect("A one-call batch pauses without releasing a partially filled instance",
                   first["http_calls"] == 1 and not drafts1)
            duplicate, no_drafts = generate_batch("cosmetic_repeat", lambda limit: client_factory(limit, punctuation_retry=True))
            attempts = json.loads((shared / "runs" / "cosmetic_repeat" / "batches.jsonl").read_text())["attempts"]
            expect("A punctuation-only retry across runs consumes its call but earns no wording slot",
                   duplicate["http_calls"] == 1 and not no_drafts
                   and [entry["disposition"] for entry in attempts] == ["accepted", "duplicate"])
            second, drafts2 = generate_batch("second")
            expect("A fresh batch recovers the first accepted variant",
                   second["http_calls"] == 1 and len(drafts2) == 2
                   and len({q["semantic_instance_id"] for q in drafts2}) == 1
                   and len({q["question"] for q in drafts2}) == 2)
            config.update(batch_size=3, max_http_calls=2 * len(rows), max_batches_per_run=0)
            final, drafts3 = generate_batch("remaining")
            expect("The remaining instances each consume their own two calls",
                   final["http_calls"] == 2 * (len(rows)-1) and len(drafts3) == 2 * (len(rows)-1))
            expect("Completed instances are absent from later accepted batches",
                   {q["semantic_instance_id"] for q in drafts2}.isdisjoint(q["semantic_instance_id"] for q in drafts3))
            expect("Every eligible instance reaches its target exactly once",
                   len({(q["semantic_instance_id"], normalized_question(q["question"])) for q in drafts2 + drafts3})
                   == 2 * len(rows))
            # Take examples only from the completed two-wording campaign, before recovery tests.
            example_drafts = drafts2 + drafts3
            by_profile = defaultdict(list)
            for draft in example_drafts:
                by_profile[draft["profile_id"]].append(draft)
            expect("Every synthetic profile has two accepted texts with different word sequences and the same answer",
                   set(by_profile) == {raw["profile_id"] for _, raw in rows}
                   and all(len(pair) == 2 and pair[0]["answer"] == pair[1]["answer"]
                           and re.findall(r"\w+", pair[0]["question"].casefold()) !=
                               re.findall(r"\w+", pair[1]["question"].casefold())
                           for pair in by_profile.values()))
            last, empty = generate_batch("all_done")
            expect("Resuming a completed campaign uses zero calls and writes zero additional QA",
                   last["http_calls"] == 0 and not empty and sum(payload_counts.values()) == 2 * len(rows) + 1)

            # Simulate losing the runtime after the provider response is saved, then before it is saved.
            config.update(requested_count=1, attempt_budget=3, batch_size=1, max_batches_per_run=1,
                          max_http_calls=1, max_total_api_calls=1)
            with patch.dict(generate_inventory_batch.__globals__, _finish_received_attempt=Mock(
                    side_effect=KeyboardInterrupt("synthetic stop before local validation"))):
                try:
                    generate_batch("after_response")
                except KeyboardInterrupt:
                    pass
                else:
                    raise AssertionError("Expected a stopped validation")
            recovered, saved = generate_batch("response_recovered")
            expect("A recorded response is recovered and validated without a second paid request",
                   recovered["http_calls"] == 0 and len(saved) == 1
                   and recovered["generation"]["campaign_reservations"] == 1)
            stopped, empty = generate_batch("cap_reached")
            expect("A cumulative campaign cap survives fresh sessions",
                   stopped["http_calls"] == 0 and not empty and stopped["generation"]["campaign_reservations"] == 1)

            config["attempt_budget"] = 2  # New isolated campaign.
            def interrupted_client(limit):
                client = Mock(calls=0)
                def lose_response(*args):
                    client.calls += 1
                    raise KeyboardInterrupt("simulated unknown provider response")
                client.generate.side_effect = lose_response
                return client
            try:
                generate_batch("lost_response", interrupted_client)
            except KeyboardInterrupt:
                pass
            else:
                raise AssertionError("Expected an interrupted dispatch")
            resumed, empty = generate_batch("unknown_held")
            unknown = [json.loads(line) for line in (shared / "runs" / "lost_response" /
                       "uncertain_requests.jsonl").read_text().splitlines()]
            expect("An unknown provider submission is held and consumes its reserved budget",
                   resumed["http_calls"] == 0 and len(unknown) == 1 and resumed["generation"]["held_instances"] == 1
                   and not empty and resumed["generation"]["campaign_reservations"] == 1)

            # Exercise both user-facing modes on the same output root and verified V13 build.
            published = dict(source, data_commit="4"*40, registry_commit="5"*40, files_manifest_sha256="6"*64,
                             verified_digests={}, verified_references={})
            entry = dict(config, mode="inventory_only", requested_count=1, attempt_budget=2,
                         tasks=[], checkpoint_recipes=2, output_dir=str(root/"entry"),
                         cache_dir=str(root/"entry-cache"), s8_dir=str(root/"entry-s8"),
                         batch_size=2, max_http_calls=2, max_total_api_calls=2)
            no_client = Mock(side_effect=AssertionError("Inventory mode requested paid generation"))
            components = {"validation": text_sha("test-compiled-parser")}
            stage_order = []

            def prepare_entry(*args, **kwargs):
                stage_order.append("prepare")
                return ({}, {}, {}), {"metadata": {}, "visuals": {}}

            def make_entry_client(limit, **kwargs):
                stage_order.append("client")
                return client_factory(limit)

            settings = dict(entry, contract_manifest_path=None, refresh_github_main=False,
                            s8_storage_root=None)
            forbidden_io = Mock(side_effect=AssertionError("Execution re-entered file admission"))
            snapshots = {}
            for mode in ("inventory_only", "real_openai"):
                admissions = {"open_campaign_release": Mock(return_value=registry),
                              "load_verified_sector1": Mock(return_value=published),
                              "resolve_real_runtime_components": Mock(return_value={"s8": "test-reader"}),
                              "prepare_real_s8_inputs": Mock(side_effect=prepare_entry)}
                with patch.dict(import_sector2_inputs.__globals__, admissions,
                                OpenAIQuestionClient=no_client, _colab_key=no_client):
                    snapshot = import_sector2_inputs(dict(settings, mode=mode))
                snapshots[mode] = snapshot
                expect(f"{mode} imports V13, S1 and S8 once before any provider call",
                       all(stage.call_count == 1 for stage in admissions.values())
                       and snapshot["registry"] is registry and snapshot["source"] is published
                       and snapshot["s8_context"] is not None and snapshot["s8_vm_sync"] is not None
                       and no_client.call_count == 0)
            imported = snapshots["inventory_only"]
            stubs = {name: forbidden_io for name in admissions}
            stubs.update(_clinical_s8_corpus=Mock(return_value=object()),
                         _verified_recipe_stream=recipe_stream, _colab_key=no_client, OpenAIQuestionClient=no_client)
            with patch.dict(execute_sector2.__globals__, stubs):
                scanned = execute_sector2(settings, imported, components)
            expect("The production inventory entry inspects all fixtures without opening an API client",
                   no_client.call_count == 0 and scanned["inventory"]["complete"] and scanned["http_calls"] == 0
                   and scanned["inventory"]["eligible_instances"] == len(rows))
            after_scan = evaluations.copy()
            forbidden_reader = Mock(side_effect=AssertionError("A completed full inventory was reopened for binding"))
            stubs.update(_verified_recipe_stream=forbidden_reader, _colab_key=lambda: "offline-test",
                         OpenAIQuestionClient=make_entry_client)
            with patch.dict(execute_sector2.__globals__, stubs):
                generated = execute_sector2(dict(settings, mode="real_openai"), imported, components)
            expect("Generation consumes the same Section 1 import without new V13, S1 or S8 checks",
                   stage_order == ["prepare", "prepare", "client"] and forbidden_io.call_count == 0
                   and stubs["_clinical_s8_corpus"].call_count == 1)
            expect("Switching from inventory_only to real_openai reuses the same published census",
                   generated["build_id"] == scanned["build_id"] and generated["inventory_database"] == scanned["inventory_database"]
                   and evaluations == after_scan and forbidden_reader.call_count == 0)
            expect("The real batch entry samples its saved backlog within the configured two-call budget",
                   generated["http_calls"] == 2 and generated["unique_drafts_added"] == 2
                   and generated["generation"]["campaign_reservations"] == 2)
        finally:
            db.close()
    return {"checks_passed": len(checks), "checks": checks, "example_drafts": example_drafts}


### 8 · S8 transfer and VM integrity

Local fixtures exercise cold and warm imports across the 5,000-file boundary. They check the number of on-disk digest evaluations, repair after failed copies, and consumption of verified metadata.


In [12]:
def run_s8_transfer_regressions():
    """Exercise the production VM transfer and SHA checks using temporary files and in-memory sources."""
    checks = []

    def expect(name, condition):
        if not condition:
            raise AssertionError(name)
        checks.append(name)

    def record(name, content):
        return {"relative_path": name, "sha256": hashlib.sha256(content).hexdigest(),
                "size_bytes": len(content)}

    def run(records, destination, copies, **settings):
        requested, events = [], []

        def open_source(entry):
            name = entry["relative_path"]
            requested.append(name)
            return BytesIO(copies(name))

        summary = sync_s8_files(records, destination, open_source,
                                progress=lambda *args: events.append(args), **settings)
        return summary, requested, events

    with tempfile.TemporaryDirectory(prefix="mmvqa-s8-transfer-") as workspace:
        root = Path(workspace)
        destination = root / "vm"
        destination.mkdir()
        content = b"test S8 evidence"
        seed = root / "seed.dat"
        seed.write_bytes(content)
        entries = [record(f"objects/{number:05d}.bin", content) for number in range(5001)]
        (destination / "objects").mkdir()
        for entry in entries[:-2]:
            os.link(seed, destination / entry["relative_path"])
        expected_missing = [entry["relative_path"] for entry in entries[-2:]]
        stat_before = (destination / entries[0]["relative_path"]).stat()

        with count_file_hashes() as hashes:
            summary, fetched, events = run(entries, destination, lambda name: content)
        expect("Each of 5,001 physical VM files receives exactly one successful digest, including cold copies",
               hashes[entries[0]["sha256"]] == 5001)
        expect("The default S8 batch size processes 5,001 manifest entries in two batches",
               summary["files"] == summary["verified"] == 5001 and summary["batches"] == 2)
        expect("Missing artifacts on opposite sides of a batch boundary are the only transfers",
               Counter(fetched) == Counter(expected_missing) and summary["copied"] == 2
               and summary["reused"] == 4999 and summary["repaired"] == summary["retried"] == 0)
        for phase in ("copy", "verify"):
            updates = [(batch, completed, length) for stage, batch, completed, length, _ in events
                       if stage == phase]
            expect(f"The {phase} progress reports a complete batch of 5,000 and a final batch of one",
                   {(batch, length) for batch, _, length in updates} == {(1, 5000), (2, 1)}
                   and (1, 5000, 5000) in updates and (2, 1, 1) in updates
                   and all(0 <= completed <= length for _, completed, length in updates))
        stat_after = (destination / entries[0]["relative_path"]).stat()
        expect("A verified file keeps its existing bytes, inode and modification timestamp",
               (stat_before.st_ino, stat_before.st_mtime_ns) == (stat_after.st_ino, stat_after.st_mtime_ns)
               and (destination / entries[0]["relative_path"]).read_bytes() == content)
        expect("Both downloaded artifacts exist at their distinct manifest paths on the VM",
               all((destination / name).read_bytes() == content for name in expected_missing))

        small = root / "repairs"
        small.mkdir()
        payloads = {"keep.bin": b"keep", "missing.bin": b"new!", "bad.bin": b"good"}
        small_entries = [record(name, data) for name, data in payloads.items()]
        (small / "keep.bin").write_bytes(payloads["keep.bin"])
        (small / "bad.bin").write_bytes(b"oops")  # Same length as the published content.
        attempts = Counter()

        def flaky_copy(name):
            attempts[name] += 1
            if name == "bad.bin" and attempts[name] == 1:
                return b"oops"
            return payloads[name]

        good = {row["relative_path"]: row["sha256"] for row in small_entries}
        wrong_sha = hashlib.sha256(b"oops").hexdigest()
        with count_file_hashes() as hashes:
            fixed, _, _ = run(small_entries, small, flaky_copy)
        expect("A repair hashes each valid file once and each distinct damaged copy once",
               all(hashes[h] == 1 for h in good.values()) and hashes[wrong_sha] == 2)
        expect("A local SHA mismatch is detected even when its byte count is correct",
               fixed["repaired"] == 1 and (small / "bad.bin").read_bytes() == payloads["bad.bin"])
        expect("A corrupt replacement is retried only for its own path",
               attempts == Counter({"bad.bin": 2, "missing.bin": 1})
               and fixed["retried"] == 1 and fixed["copied"] == 1 and fixed["reused"] == 1)
        expect("All files in the repaired batch match their published SHA-256 on disk",
               fixed["verified"] == len(small_entries) and all(
                   hashlib.sha256((small / row["relative_path"]).read_bytes()).hexdigest() == row["sha256"]
                   for row in small_entries))

        def unexpected_transfer(name):
            raise AssertionError("A valid local file was fetched again: " + name)

        resumed, fetched, _ = run(small_entries, small, unexpected_transfer)
        expect("Reopening a complete cache verifies every file without any transfer",
               not fetched and resumed["verified"] == resumed["reused"] == len(small_entries))

        untrusted = dict(small_entries[0])
        untrusted.pop("sha256")
        transfer = Mock(side_effect=AssertionError("A file without an expected digest reached the source"))
        try:
            sync_s8_files([untrusted], small, transfer, progress=lambda *args: None)
        except (ValueError, KeyError):
            expect("A manifest entry without an authoritative digest stops before any transfer",
                   not transfer.called)
        else:
            raise AssertionError("A missing expected SHA-256 was accepted")

        (small / "bad.bin").write_bytes(b"prev")
        before = (small / "bad.bin").stat()
        broken_calls = []

        def permanently_bad(entry):
            broken_calls.append(entry["relative_path"])
            return BytesIO(b"fail")

        try:
            sync_s8_files(small_entries, small, permanently_bad, max_attempts=3,
                          progress=lambda *args: None)
        except RuntimeError as exc:
            expect("An unrecoverable checksum mismatch identifies the failed path",
                   "bad.bin" in str(exc))
        else:
            raise AssertionError("Sync reported success despite three invalid copies")
        after = (small / "bad.bin").stat()
        expect("Three failed copies do not replace the previous file or refetch verified neighbors",
               broken_calls == ["bad.bin"] * 3 and (small / "bad.bin").read_bytes() == b"prev"
               and (before.st_ino, before.st_mtime_ns) == (after.st_ino, after.st_mtime_ns)
               and (small / "keep.bin").read_bytes() == payloads["keep.bin"]
               and (small / "missing.bin").read_bytes() == payloads["missing.bin"])
        recovered, calls, _ = run(small_entries, small, payloads.__getitem__)
        expect("Resuming after a failed sync repairs only the outstanding artifact",
               calls == ["bad.bin"] and recovered["verified"] == 3 and recovered["reused"] == 2)

        # Corrupt bytes in the destination stream while the remote source supplies valid bytes.
        # Only a digest of the written VM file can detect this fault.
        (small / "bad.bin").unlink()
        real_copy = shutil.copyfileobj
        disk_copies = []

        def disk_fault_once(source, target, **options):
            real_copy(source, target, **options)
            disk_copies.append(target.name)
            if len(disk_copies) == 1:
                target.seek(0)
                target.write(b"oops")

        with patch.object(shutil, "copyfileobj", disk_fault_once):
            durable, calls, _ = run(small_entries, small, payloads.__getitem__)
        expect("A local write fault is caught after staging even if the source stream was correct",
               calls == ["bad.bin", "bad.bin"] and len(disk_copies) == 2
               and durable["retried"] == 1 and (small / "bad.bin").read_bytes() == b"good")

        # A short HTTP response can raise IncompleteRead between two chunks.
        (small / "missing.bin").unlink()
        streams = []

        class TruncatedStream(BytesIO):
            def read(self, size=-1):
                first = super().read(size)
                if first:
                    return first
                raise IncompleteRead(b"", 2)

        def restore_after_truncation(entry):
            name = entry["relative_path"]
            streams.append(name)
            return TruncatedStream(b"ne") if len(streams) == 1 else BytesIO(payloads[name])

        truncated = sync_s8_files(small_entries, small, restore_after_truncation, progress=lambda *args: None)
        expect("An interrupted HTTP response retries only its incomplete file",
               streams == ["missing.bin"] * 2 and truncated["retried"] == 1 and truncated["reused"] == 2
               and (small / "missing.bin").read_bytes() == payloads["missing.bin"])

        bars = []

        def make_bar(**options):
            bar = Mock()
            bars.append((options, bar))
            return bar

        with patch.dict(sync_s8_files.__globals__, tqdm=make_bar):
            sync_s8_files(small_entries, small, unexpected_transfer, batch_size=2)
        expect("The default Colab progress renderer opens and closes two bars per batch",
               [options["total"] for options, _ in bars] == [2, 2, 1, 1]
               and all(bar.close.call_count == 1 and sum(call.args[0] for call in bar.update.call_args_list)
                       == options["total"] for options, bar in bars))
    return {"checks_passed": len(checks), "checks": checks}


def run_s8_visual_adapter_regressions():
    """Check original and archived S8 images at the same paths used by the production adapter."""
    checks = []

    def expect(name, condition):
        if not condition:
            raise AssertionError(name)
        checks.append(name)

    def archive_member(name, data):
        return {"relative_path": name, "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest()}

    with tempfile.TemporaryDirectory(prefix="mmvqa-s8-media-") as workspace:
        root = Path(workspace)
        upstream = root / "mounted-data"
        sector8 = "outputs/phase2/sector8"
        original = "data/labelled_images/Example/0001.jpg"
        derived = "products/annotation_crops/0001.png"
        original_bytes = b"synthetic source file"
        crop_bytes = b"synthetic crop file"
        original_record = archive_member(original, original_bytes)
        crop_record = archive_member(derived, crop_bytes)
        original_file = upstream / original
        original_file.parent.mkdir(parents=True)
        original_file.write_bytes(original_bytes)
        tar_relpath = "archives/annotation_crops.tar"
        tar_path = upstream / sector8 / tar_relpath
        tar_path.parent.mkdir(parents=True)
        with tarfile.open(tar_path, mode="w") as writer:
            member = tarfile.TarInfo(derived)
            member.size = len(crop_bytes)
            writer.addfile(member, BytesIO(crop_bytes))
        archive = archive_member(tar_relpath, tar_path.read_bytes())
        archive["members"] = [crop_record]
        source_row = {"image_relpath": original, "sha256": original_record["sha256"],
                      "source_sha256": original_record["sha256"], "image_size_bytes": len(original_bytes)}
        corpus = Mock(
            source_config={"storage_root": str(upstream), "sector8_relative_root": sector8},
            source_snapshot={"repository": "tests/demo", "resolved_commit": "a" * 40},
            sources_by_path={original: source_row},
            visual_by_artifact={
                "source1": dict(original_record, storage="source"),
                "crop1": dict(crop_record, storage="derived"),
            },
            benchmark=Mock(archive_manifest={"archives": [archive]}),
        )
        vm = root / "vm"
        result = sync_s8_visuals(corpus, vm, storage_root=upstream, progress=lambda *args: None)
        cache = Path(result["directory"])
        expect("The visual adapter materializes the original from its mounted source path",
               (cache / original).read_bytes() == original_bytes)
        expect("The visual adapter materializes a derived file from the named TAR member",
               (cache / derived).read_bytes() == crop_bytes and not (upstream / sector8 / derived).exists())
        expect("The same original in two S8 manifest tables is staged once",
               result["files"] == result["copied"] == result["verified"] == 2)
        (cache / derived).write_bytes(b"X" * len(crop_bytes))
        loose_copy = upstream / sector8 / derived
        loose_copy.parent.mkdir(parents=True)
        loose_copy.write_bytes(b"Y" * len(crop_bytes))
        fixed = sync_s8_visuals(corpus, vm, storage_root=upstream, progress=lambda *args: None)
        expect("A corrupted loose product recovers from its own published TAR member",
               fixed["repaired"] == fixed["retried"] == fixed["reused"] == 1
               and (cache / derived).read_bytes() == crop_bytes)
        shutil.rmtree(upstream)
        resumed = sync_s8_visuals(corpus, vm, storage_root=upstream, progress=lambda *args: None)
        expect("A verified visual cache supports a restart without access to the mounted source",
               result["directory"] == resumed["directory"] and resumed["verified"] == resumed["reused"] == 2
               and resumed["copied"] == resumed["repaired"] == resumed["retried"] == 0)
    return {"checks_passed": len(checks), "checks": checks}


def run_s8_metadata_sync_regressions():
    """Exercise the storage index, a direct GitHub object and a split object without networking."""
    index = "configs/sector8_github_storage.json"
    hosted_root = "outputs/phase2/sector8"

    class LocalStore:
        # Only the cache naming and index interfaces normally supplied by the pinned reader.
        def __init__(self, repo, ref, cache_dir):
            self.repo = repo
            self.consumed = {}
            self.cache_dir = Path(cache_dir)

        def _cache_path(self, section, name):
            return self._root / section / name

        def _validate_index(self, index_document):
            return {entry["relative_path"]: entry for entry in index_document["files"]}

    env = {"S8Store": LocalStore, "INDEX_PATH": index, "S8_ROOT": hosted_root,
           "safe_relative_path": lambda name: name}

    def record(name, data, **fields):
        return {"relative_path": name, "bytes": len(data), "sha256": hashlib.sha256(data).hexdigest(), **fields}

    part_a = record("parts/a.bin", b"A1", ordinal=0, offset=0)
    part_b = record("parts/b.bin", b"B2", ordinal=1, offset=2)
    table = record("tables/evidence.parquet", b"A1B2", storage="parts", parts=[part_a, part_b])
    direct = record("manifest.json", b"{}", storage="direct", github_relative_path="objects/manifest.json")
    document = {"schema_version": 1, "mode": "exact_byte_parts", "source_catalog_sha256": table["sha256"],
                "files": [table, direct]}
    remote = {index: json.dumps(document).encode(), part_a["relative_path"]: b"A1",
              part_b["relative_path"]: b"B2", direct["github_relative_path"]: b"{}"}
    pin = {"repository": "tests/demo", "s8_root": hosted_root, "resolved_commit": "a" * 40,
           "storage_index": record(index, remote[index]), "source_catalog_sha256": table["sha256"],
           "consumed_artifacts": [table, {k: v for k, v in direct.items() if k != "github_relative_path"}]}
    calls, checks = [], []
    def expect(name, condition):
        if not condition:
            raise AssertionError(name)
        checks.append(name)

    def opened(name):
        calls.append(name)
        if name == part_b["relative_path"] and calls.count(name) == 1 and not ready:
            return BytesIO(b"XX")
        return BytesIO(remote[name])

    with tempfile.TemporaryDirectory(prefix="mmvqa-s8-metadata-") as root:
        def stage():
            return prepare_s8_metadata(env, pin, root, progress=lambda *args: None, open_remote=opened)

        ready = False
        with count_file_hashes() as hashes:
            store = stage()
        expected_digests = [pin["storage_index"]["sha256"], table["sha256"], direct["sha256"],
                            part_a["sha256"], part_b["sha256"]]
        expect("The S8 index, each successful part and each logical file are hashed once when imported",
               all(hashes[h] == 1 for h in expected_digests))
        expected = Counter(remote.keys())
        expected[part_b["relative_path"]] += 1
        expect("A corrupt GitHub part retries only that part while staging metadata", Counter(calls) == expected)
        expect("The merged logical table has the published bytes after assembly",
               store._cache_path("logical", table["relative_path"]).read_bytes() == b"A1B2"
               and store.sector2_sync_summary["logical_metadata"]["verified"] == 2)
        with count_file_hashes() as consumer_hashes:
            store.load(table["relative_path"])
            parsed = store.read_json(direct["relative_path"])
            store.load(table["relative_path"])
        expect("The clinical reader uses admitted paths and captured JSON without repeating any digest",
               not consumer_hashes and parsed == {})
        calls.clear()
        ready = True
        with count_file_hashes() as hashes:
            store = stage()
        expect("A fresh import on a warm VM hashes each final logical file once, without rereading its parts",
               all(hashes[h] == 1 for h in expected_digests[:3])
               and not any(hashes[h] for h in expected_digests[3:]))
        expect("A complete local S8 index and both logical objects reopen without network",
               not calls and store.sector2_sync_summary["storage_index"]["reused"] == 1
               and store.sector2_sync_summary["logical_metadata"]["reused"] == 2)
        store._cache_path("logical", table["relative_path"]).unlink()
        store._cache_path("parts", part_b["sha256"] + ".bin").write_bytes(b"XX")
        store = stage()
        expect("Rebuilding a missing logical file fetches only its damaged part",
               calls == [part_b["relative_path"]]
               and store._cache_path("logical", table["relative_path"]).read_bytes() == b"A1B2")
        calls.clear()
        store._cache_path("logical", direct["relative_path"]).write_bytes(b"xx")
        store._cache_path("publication", index).write_bytes(b"?")
        stage()
        expect("A bad pinned index and one corrupt direct file leave the multipart table untouched",
               Counter(calls) == Counter([index, direct["github_relative_path"]]))
    return {"checks_passed": len(checks), "checks": checks}


### 9 · Run the offline suites

The suites use the registry verified in setup. Network access is blocked throughout the tests, including the mocked GitHub and OpenAI transports. Synthetic evidence, audit databases and interrupted jobs are temporary. The report identifies the tested engine, the contract source, the GitHub revision when applicable, and the wording-fixture results.

The CSV contains just the accepted drafts from one completed offline campaign, with the reference answers produced by the engine. The console shows each question and the absolute CSV path. The separate interruption tests and the prewritten wording benchmark are reported only in the test summary.


In [13]:
with offline_only():
    registry = prepared_registry
    contracts = registry["contracts"]
    profiles = [row["profile_id"] for _, row in fixture_evidence_rows()]
    if len(set(profiles)) != 8 or len(profiles) != 8:
        raise AssertionError(f"Expected eight distinct synthetic profiles, observed {profiles!r}")

    suite_results = {}
    for suite, check in (
        ("Section 1 execution order", lambda: run_import_layout_regressions(production_path)),
        ("S8 transfer and local SHA-256", lambda: run_s8_transfer_regressions()),
        ("S8 storage index and multipart transfer", lambda: run_s8_metadata_sync_regressions()),
        ("S8 original and archived visuals", lambda: run_s8_visual_adapter_regressions()),
        ("S8 published references", lambda: run_s8_reference_regressions()),
        ("Evidence and semantic validation", lambda: run_fixture_regressions(contracts)),
        ("Audit and authorization", lambda: run_audit_regressions(registry)),
        ("OpenAI adapter with an in-memory response", lambda: test_offline_provider(contracts)),
        ("Single-pass S1 publication import", lambda: run_s1_import_regressions(registry)),
        ("Manifest integrity", lambda: run_manifest_regressions(registry["staged_directory"])),
        ("Census, batch and checkpoint resumption", lambda: run_inventory_batch_regressions(registry)),
    ):
        suite_results[suite] = check()
        print(f"{suite}: {suite_results[suite]['checks_passed']} checks passed")

    test_required_fixture_outcomes(contracts)
    wording_tests = benchmark_contract_wording(contracts)
    wording_failures = [case for case in wording_tests["cases"] if case["outcome"] == "failed"]
    if wording_failures:
        raise AssertionError("V13 wording benchmark failures:\n" + json.dumps(wording_failures, indent=2))

test_report = {
    "status": "passed", "test_network_access": "blocked", "contract_source": CONTRACT_SOURCE,
    "contract_revision": registry["registry_commit"], "manifest_source": preparation_manifest_source,
    "production_notebook_sha256": production_sha256, "components_sha256": component_hashes,
    "contracts_manifest_sha256": registry["manifest_sha256"],
    "verified_contract_files": len(registry["contract_sha256"]), "synthetic_profiles": profiles,
    "suite_check_counts": {name: result["checks_passed"] for name, result in suite_results.items()},
    "wording_benchmark": wording_tests["summary"],
    "missing_or_mismatched_required_check_regression": "passed",
}
print(json.dumps(test_report, indent=2))


example_drafts = suite_results["Census, batch and checkpoint resumption"]["example_drafts"]
csv_path = (default_work / "mmvqa_offline_qa_examples.csv").resolve()
columns = ["test_scenario", "task", "profile_id", "semantic_instance_id", "variant", "question", "reference_answer"]
csv_buffer = io.StringIO(newline="")
writer = csv.DictWriter(csv_buffer, fieldnames=columns)
writer.writeheader()
variants = Counter()
for draft in example_drafts:
    instance_id = draft["semantic_instance_id"]
    variants[instance_id] += 1
    answer = draft["answer"]
    writer.writerow({
        "test_scenario": "synthetic_offline_two_per_instance",
        "task": draft["task_id"], "profile_id": draft["profile_id"], "semantic_instance_id": instance_id,
        "variant": variants[instance_id], "question": draft["question"],
        "reference_answer": answer if isinstance(answer, str) else json.dumps(answer, ensure_ascii=False),
    })
_write_atomic(csv_path, csv_buffer.getvalue().encode("utf-8-sig"))
print(f"Saved {len(example_drafts)} accepted offline examples from {len(variants)} synthetic instances to {csv_path}")
for draft in example_drafts:
    print(f'{draft["profile_id"]} | {draft["question"]}')


Section 1 execution order: 3 checks passed
S8 transfer and local SHA-256: 19 checks passed
S8 storage index and multipart transfer: 8 checks passed
S8 original and archived visuals: 5 checks passed
S8 published references: 10 checks passed
Evidence and semantic validation: 68 checks passed
Audit and authorization: 22 checks passed
OpenAI adapter with an in-memory response: 3 checks passed
Single-pass S1 publication import: 5 checks passed


Git · contracts.v13.manifest.json · batch 1 · files processed for transfer:   0%|          | 0/1 [00:00<?, ?fi…

Git · contracts.v13.manifest.json · batch 1 · verified VM files:   0%|          | 0/1 [00:00<?, ?file/s]

V13 contracts · batch 1 · files processed for transfer:   0%|          | 0/14 [00:00<?, ?file/s]

V13 contracts · batch 1 · verified VM files:   0%|          | 0/14 [00:00<?, ?file/s]

Git · contracts.v13.manifest.json · batch 1 · files processed for transfer:   0%|          | 0/1 [00:00<?, ?fi…

... (88 outputs left out)

V13 detached manifest · batch 1 · verified VM files:   0%|          | 0/1 [00:00<?, ?file/s]

V13 detached manifest · batch 1 · files processed for transfer:   0%|          | 0/1 [00:00<?, ?file/s]

V13 detached manifest · batch 1 · verified VM files:   0%|          | 0/1 [00:00<?, ?file/s]

Manifest integrity: 42 checks passed
S8 inventory: 0/8 recipes committed.
S8 inventory: 1/8 recipes committed.
S8 inventory: 8/8 recipes committed.
Recipes examined: 8 | Unique eligible instances: 8
evidence_frame_localization 1 instances; 2 target questions
finding_category 1 instances; 2 target questions
finding_identification 2 instances; 4 target questions
observed_anchor_order 1 instances; 2 target questions
optional_spatial_localization 1 instances; 2 target questions
temporal_relative_evidence_localization 1 instances; 2 target questions
verified_reference_comparison 1 instances; 2 target questions
Corpus question target: 16
S8 inventory: 0/8 recipes committed.
S8 inventory: 8/8 recipes committed.
Recipes examined: 8 | Unique eligible instances: 8
evidence_frame_localization 1 instances; 1 target questions
finding_category 1 instances; 1 target questions
finding_identification 2 instances; 2 target questions
observed_anchor_order 1 instances; 1 target questions
optional_spatial_